# Journal figures and tables

Reusable workspace for figures and tables for the journal draft.

In [ ]:
from __future__ import annotations

import gzip
import json
import pickle
import sys
from collections import Counter, defaultdict
from datetime import UTC, datetime
from importlib import resources
from pathlib import Path
from typing import Any

import amd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymatgen.analysis.prototypes as prototypes
import umap
from amd.io import periodicset_from_pymatgen_structure
from IPython.display import Markdown, display
from pymatgen.analysis.prototypes import (
    WYCKOFF_MULTIPLICITY_DICT,
    WYCKOFF_POSITION_RELAB_DICT,
)
from pymatgen.symmetry.groups import sg_symbol_from_int_number
from sklearn.metrics import pairwise_distances_chunked
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate
from tqdm.auto import tqdm

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    notebooks_dir = path / "notebooks"
    if (notebooks_dir / "notebook_utils.py").exists():
        if str(notebooks_dir) not in sys.path:
            sys.path.insert(0, str(notebooks_dir))
        break
else:
    raise RuntimeError("Could not find notebooks directory")

from notebook_constants import (  # noqa: E402
    CATEGORY_LABELS,
    CATEGORY_ORDER,
    CRYSTAL_SYSTEM_ORDER,
    METASTABLE_EHULL_MAX,
    MODEL_DISPLAY_NAMES,
    SIMPLE_CATEGORY_LABELS,
    SIMPLE_CATEGORY_ORDER,
    SOURCE_ORDER,
    WYCKOFF_REPR_FILE,
)
from notebook_utils import (  # noqa: E402
    classify_model,
    crystal_system_from_spg_num,
    entries_to_frame,
    find_repo_root,
    load_pickle_gz,
    missing_required_paths,
    required_paths,
)
from plot_style import (  # noqa: E402
    BLACK,
    CATEGORY_COLORS,
    GRAY,
    PALETTE,
    WHITE,
    apply_plot_style,
)

ROOT = find_repo_root()
INPUT_DIR = ROOT / "input"
RAW_RESULTS_DIR = ROOT / "results" / "raw"
ANALYSIS_RESULTS_DIR = ROOT / "results" / "analysis"
SIMPLE_CATEGORY_COLORS = {
    "1": CATEGORY_COLORS["1"],
    "2": CATEGORY_COLORS["2-1"],
    "3": CATEGORY_COLORS["3"],
}
EMBEDDING_PROJECTION_CACHE_DIR = ROOT / "notebooks" / ".cache" / "embedding_projections"

for import_path in (ROOT,):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from src._subst_cost import subst_cost_mod_petti  # noqa: E402

apply_plot_style()

CLASSIFICATION_PATH_KEYS = (
    "generated_structures",
    "relax_infos",
    "relaxed_ehull",
    "smact_validity",
    "direct_sm",
    "relaxed_sm_anon_matches",
    "relaxed_wyckoff_matches",
    "wyckoff_repr",
)

FIG_WIDTH = 7.18
FONT_LL = 12
FONT_L = 10
FONT_M = 8
PAD_L = 10
PAD_M = 8
PAD_S = 6
PAD_SS = 4

## Load classifications

In [ ]:
def discover_models() -> list[str]:
    gen_dir = INPUT_DIR / "gen" / "preprocessed"
    if not gen_dir.exists():
        return []
    return sorted(path.name for path in gen_dir.iterdir() if path.is_dir())


def model_required_paths(model: str) -> dict[str, Path]:
    return required_paths(model, INPUT_DIR, RAW_RESULTS_DIR, CLASSIFICATION_PATH_KEYS)


models = discover_models()
missing_files = pd.DataFrame(
    record
    for model in models
    for record in missing_required_paths(model_required_paths(model), model=model)
)
missing_models = set(missing_files["model"]) if not missing_files.empty else set()
complete_models = [model for model in models if model not in missing_models]

classifications = (
    pd.concat(
        [
            classify_model(
                model,
                model_required_paths(model),
                include_category_label=True,
                include_crystal_system=True,
                include_simple_category=True,
                simple_category_labels=SIMPLE_CATEGORY_LABELS,
                simple_category_order=SIMPLE_CATEGORY_ORDER,
            )
            for model in complete_models
        ],
        ignore_index=True,
    )
    if complete_models
    else pd.DataFrame(
        columns=[
            "model",
            "gen_idx",
            "category",
            "category_label",
            "crystal_system",
            "simple_category",
            "simple_category_label",
            "is_direct_sm_match",
            "has_relaxed_sm_anon_match",
            "has_relaxed_wyckoff_match",
            "ehull_relaxed",
            "is_relax_converged",
            "is_metastable",
            "is_smact_valid",
            "is_metastable_smact_valid",
        ]
    )
)

display(Markdown(f"**Complete models:** {', '.join(complete_models) or 'none'}"))
if missing_files.empty:
    display(Markdown("**Skipped models:** none"))
else:
    display(Markdown("**Skipped models with missing files:**"))
    display(missing_files)

## Metastable SMACT-valid novelty table

Percentages are computed within samples where relaxation converged, relaxed `Ehull <= 0.1 eV/atom`, and composition is SMACT-valid. Detailed category-2 variants are aggregated as `subst match`.

In [ ]:
def build_novelty_percent_table(classifications: pd.DataFrame) -> pd.DataFrame:
    columns = ["n", "Duplicate (%)", "Substituted (%)", "Unmatched (%)"]
    if classifications.empty:
        return pd.DataFrame(columns=columns)

    subset = classifications[classifications["is_metastable_smact_valid"]].copy()
    counts = (
        subset.groupby(["model", "simple_category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [complete_models, SIMPLE_CATEGORY_ORDER],
        names=["model", "simple_category"],
    )
    summary = counts.reindex(full_index, fill_value=0).reset_index()
    totals = summary.groupby("model")["count"].transform("sum")
    summary["percent"] = np.where(totals > 0, 100 * summary["count"] / totals, 0.0)
    table = summary.pivot(
        index="model",
        columns="simple_category",
        values="percent",
    ).rename(columns=SIMPLE_CATEGORY_LABELS)
    table = table.rename(columns={label: f"{label} (%)" for label in table.columns})
    table.insert(
        0, "n", subset.groupby("model").size().reindex(complete_models, fill_value=0)
    )
    table = table.loc[complete_models, columns]

    percent_sum = table[["Duplicate (%)", "Substituted (%)", "Unmatched (%)"]].sum(
        axis=1
    )
    assert np.allclose(percent_sum[table["n"] > 0], 100.0)
    return table.round(2)


journal_novelty_table = build_novelty_percent_table(classifications)
journal_novelty_table

## Category ratios by crystal system

Stacked ratios for metastable SMACT-valid samples, with detailed category-2 variants shown using hatch patterns.

In [ ]:
DETAILED_CATEGORY_HATCHES = {
    "1": "",
    "2-1": "",
    "2-2": "...",
    "2-3": "\\\\\\\\\\\\",
    "3": "",
}
DETAILED_CATEGORY_COLORS = {
    "1": CATEGORY_COLORS["1"],
    "2-1": CATEGORY_COLORS["2-1"],
    "2-2": CATEGORY_COLORS["2-1"],
    "2-3": CATEGORY_COLORS["2-1"],
    "3": CATEGORY_COLORS["3"],
}


def build_detailed_crystal_system_category_ratios(
    selected: pd.DataFrame,
) -> pd.DataFrame:
    if selected.empty:
        return pd.DataFrame(columns=["crystal_system", "category", "count", "ratio"])

    counts = (
        selected.groupby(["crystal_system", "category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [CRYSTAL_SYSTEM_ORDER, CATEGORY_ORDER],
        names=["crystal_system", "category"],
    )
    result = counts.reindex(full_index, fill_value=0).reset_index()
    totals = result.groupby("crystal_system", observed=False)["count"].transform("sum")
    result["ratio"] = np.where(totals > 0, result["count"] / totals, 0.0)
    result["category_label"] = result["category"].map(CATEGORY_LABELS)
    return result


def build_detailed_model_crystal_system_category_ratios(
    classifications: pd.DataFrame,
) -> pd.DataFrame:
    columns = ["model", "crystal_system", "category", "count", "ratio"]
    if classifications.empty:
        return pd.DataFrame(columns=columns)

    selected = classifications[classifications["is_metastable_smact_valid"]].copy()
    counts = (
        selected.groupby(["model", "crystal_system", "category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [complete_models, CRYSTAL_SYSTEM_ORDER, CATEGORY_ORDER],
        names=["model", "crystal_system", "category"],
    )
    result = counts.reindex(full_index, fill_value=0).reset_index()
    totals = result.groupby(["model", "crystal_system"], observed=False)[
        "count"
    ].transform("sum")
    result["ratio"] = np.where(totals > 0, result["count"] / totals, 0.0)
    return result[columns]


def build_tabulated_detailed_model_crystal_system_ratios(
    counts_by_system: pd.DataFrame,
) -> str:
    if counts_by_system.empty:
        return "No classifications available."

    count_table = (
        counts_by_system.pivot(
            index=["model", "crystal_system"], columns="category", values="count"
        )
        .reindex(
            index=pd.MultiIndex.from_product(
                [complete_models, CRYSTAL_SYSTEM_ORDER],
                names=["model", "crystal_system"],
            ),
            columns=CATEGORY_ORDER,
        )
        .fillna(0)
        .astype(int)
    )
    totals = count_table.sum(axis=1)
    count_table = count_table[totals > 0]
    totals = totals[totals > 0]
    if count_table.empty:
        return "No classifications available."

    ratio_table = count_table.div(totals, axis=0)
    display_table = pd.DataFrame(index=count_table.index)
    display_table["model"] = [
        MODEL_DISPLAY_NAMES.get(model, model) for model, _ in count_table.index
    ]
    display_table["crystal_system"] = [system for _, system in count_table.index]
    for category in CATEGORY_ORDER:
        display_table[CATEGORY_LABELS[category]] = [
            f"{count:,} ({ratio:.3f})"
            for count, ratio in zip(
                count_table[category], ratio_table[category], strict=True
            )
        ]
    display_table = display_table.reset_index(drop=True)
    return tabulate(display_table, headers="keys", tablefmt="github", showindex=False)


def add_stacked_bars(
    ax,
    x: np.ndarray,
    values: pd.DataFrame,
    columns: list[str],
    colors: dict[str, str],
    labels: dict[str, str],
    *,
    width: float = 0.8,
    edgecolor: str | None = None,
    linewidth: float = 0.0,
    hatches: dict[str, str] | None = None,
    antialiased: bool | None = None,
    rasterized: bool | None = None,
    snap: bool | None = None,
) -> tuple[dict[str, object], np.ndarray]:
    bottom = np.zeros(len(values), dtype=float)
    legend_handles = {}
    for column in columns:
        heights = values[column].to_numpy(dtype=float)
        bar_edgecolor = colors[column] if edgecolor == "face" else edgecolor
        legend_handles[column] = ax.bar(
            x,
            heights,
            bottom=bottom,
            width=width,
            color=colors[column],
            edgecolor=bar_edgecolor,
            linewidth=linewidth,
            hatch=(hatches or {}).get(column, ""),
            label=labels[column],
            antialiased=antialiased,
            rasterized=rasterized,
            snap=snap,
        )
        bottom += heights
    return legend_handles, bottom


def model_metastable_smact_valid_classifications(model: str) -> pd.DataFrame:
    return classifications[
        (classifications["model"] == model)
        & classifications["is_metastable_smact_valid"]
    ].copy()


mattergen_classifications = model_metastable_smact_valid_classifications("mattergen")


def plot_detailed_category_ratios_by_crystal_system(
    ax,
    counts_by_system: pd.DataFrame,
    *,
    title: str,
    show_ylabel: bool,
) -> dict[str, object]:
    ax.set_title(title, fontsize=FONT_LL, pad=PAD_L)
    if counts_by_system.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_axis_off()
        return {}

    values = (
        counts_by_system.pivot(
            index="crystal_system",
            columns="category",
            values="ratio",
        )
        .reindex(index=CRYSTAL_SYSTEM_ORDER, columns=CATEGORY_ORDER)
        .fillna(0.0)
    )
    totals = (
        counts_by_system.groupby("crystal_system", observed=False)["count"]
        .sum()
        .reindex(CRYSTAL_SYSTEM_ORDER, fill_value=0)
    )
    systems_with_data = totals[totals > 0].index.tolist()
    if not systems_with_data:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_axis_off()
        return {}

    values = values.loc[systems_with_data]
    assert np.allclose(values.sum(axis=1), 1.0)

    x = np.arange(len(values))
    legend_handles, _ = add_stacked_bars(
        ax,
        x,
        values,
        CATEGORY_ORDER,
        DETAILED_CATEGORY_COLORS,
        CATEGORY_LABELS,
        edgecolor=WHITE,
        linewidth=0.5,
        hatches=DETAILED_CATEGORY_HATCHES,
    )

    if show_ylabel:
        ax.set_ylabel("Ratio", fontsize=FONT_L, labelpad=PAD_M)
    ax.set_ylim(0, 1)
    ax.set_xticks(x, systems_with_data, rotation=40, ha="right")
    ax.tick_params(axis="both", labelsize=FONT_M, pad=PAD_SS)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return legend_handles


def plot_model_crystal_system_ratio_figure(
    models: list[str],
    *,
    layout: tuple[int, int],
    output_name: str,
    panel_tags: tuple[str, ...] | None = None,
) -> None:
    nrows, ncols = layout
    height = FIG_WIDTH * (0.45 if nrows == 1 else 0.85)
    fig, axes = plt.subplots(nrows, ncols, figsize=(FIG_WIDTH, height))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)
    legend_handles = {}

    for index, model in enumerate(models):
        row, col = divmod(index, ncols)
        selected = model_metastable_smact_valid_classifications(model)
        counts_by_system = build_detailed_crystal_system_category_ratios(selected)
        handles = plot_detailed_category_ratios_by_crystal_system(
            axes[row, col],
            counts_by_system,
            title=MODEL_DISPLAY_NAMES.get(model, model),
            show_ylabel=col == 0,
        )
        if handles:
            legend_handles = handles

    for ax in axes.ravel()[len(models) :]:
        ax.set_axis_off()

    if panel_tags is not None:
        for tag, ax in zip(panel_tags, axes.ravel(), strict=True):
            ax.text(
                -0.20,
                1.15,
                tag,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=FONT_LL,
                fontweight="bold",
                color=BLACK,
            )

    if legend_handles:
        fig.legend(
            handles=[legend_handles[category] for category in CATEGORY_ORDER],
            labels=[CATEGORY_LABELS[category] for category in CATEGORY_ORDER],
            loc="lower center",
            bbox_to_anchor=(0.5, -0.05),
            frameon=False,
            fontsize=FONT_M,
            ncol=len(CATEGORY_ORDER),
            handleheight=1.2,
            handlelength=1.5,
            handletextpad=0.5,
            columnspacing=1.0,
        )
    fig.tight_layout()

    plot_dir = ANALYSIS_RESULTS_DIR / "journal"
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        plot_dir / output_name,
        bbox_inches="tight",
    )
    plt.show()


model_crystal_system_ratios = build_detailed_model_crystal_system_category_ratios(
    classifications
)
display(
    Markdown(
        build_tabulated_detailed_model_crystal_system_ratios(
            model_crystal_system_ratios
        )
    )
)
plot_model_crystal_system_ratio_figure(
    ["mattergen", "test"],
    layout=(1, 2),
    output_name="category_ratios_by_crystal_system_mattergen_mp20_test.pdf",
    panel_tags=("(a)", "(b)"),
)
plot_model_crystal_system_ratio_figure(
    ["diffcsppp", "wyckofftransformer", "crystalite", "chemeleon2"],
    layout=(2, 2),
    output_name="category_ratios_by_crystal_system_other_models.pdf",
)

## MatterGen top-k substituted ratios

Classification ratios for metastable SMACT-valid MatterGen samples when substituted matches are limited to the top-k Wyckoff and SM-anon candidates.


In [ ]:
TOPK_VALUES = list(range(1, 6))
TOPK_MATCH_PATHS = {
    "wyckoff": RAW_RESULTS_DIR
    / "mattergen"
    / (
        "substituted_relaxed_niggli_top5_wyckoff_match_s=0.01_"
        "c=mod_petti_gen_matches.pkl.gz"
    ),
    "sm_anon": RAW_RESULTS_DIR
    / "mattergen"
    / "substituted_relaxed_niggli_top5_sm_anon_gen_matches.pkl.gz",
}
TOPK_SUBSTITUTED_COLOR = CATEGORY_COLORS["2-1"]


def load_topk_matched_indices(
    paths: dict[str, Path],
    selected_indices: set[int],
    *,
    max_k: int,
) -> dict[int, set[int]]:
    matched_by_k = {k: set() for k in range(1, max_k + 1)}
    for source, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {source} top-k match file: {path}")
        for record in load_pickle_gz(path):
            if not bool(record.get("match", False)):
                continue
            gen_idx = int(record["gen_idx"])
            if gen_idx not in selected_indices:
                continue
            rank = int(record["rank"])
            for k in range(max(rank, 1), max_k + 1):
                matched_by_k[k].add(gen_idx)
    return matched_by_k


def build_mattergen_topk_substituted_ratios(
    selected: pd.DataFrame,
    *,
    k_values: list[int] = TOPK_VALUES,
) -> pd.DataFrame:
    columns = ["k", "count", "ratio", "substituted_delta"]
    if selected.empty:
        return pd.DataFrame(columns=columns)

    selected_indices = set(int(idx) for idx in selected["gen_idx"])
    duplicate_indices = set(
        int(idx) for idx in selected.loc[selected["is_direct_sm_match"], "gen_idx"]
    )
    matched_by_k = load_topk_matched_indices(
        TOPK_MATCH_PATHS,
        selected_indices,
        max_k=max(k_values),
    )

    rows = []
    previous_substituted_ratio = np.nan
    total = len(selected_indices)
    for k in k_values:
        substituted_indices = matched_by_k[k] - duplicate_indices
        substituted_count = len(substituted_indices)
        substituted_ratio = substituted_count / total
        substituted_delta = (
            np.nan
            if np.isnan(previous_substituted_ratio)
            else substituted_ratio - previous_substituted_ratio
        )
        rows.append(
            {
                "k": k,
                "count": substituted_count,
                "ratio": substituted_ratio,
                "substituted_delta": substituted_delta,
            }
        )
        previous_substituted_ratio = substituted_ratio

    result = pd.DataFrame(rows, columns=columns)
    assert result["k"].tolist() == k_values
    assert result["ratio"].between(0.0, 1.0).all()
    assert np.all(np.diff(result["ratio"].to_numpy(dtype=float)) >= -1e-12)
    return result


def plot_mattergen_topk_substituted_ratios(ratios: pd.DataFrame) -> None:
    if ratios.empty:
        display(Markdown("No MatterGen top-k substituted ratios available."))
        return

    substituted = ratios.set_index("k").reindex(TOPK_VALUES)

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_WIDTH * 0.62))
    x = np.arange(len(substituted))
    ax.bar(
        x,
        substituted["ratio"].to_numpy(dtype=float),
        width=0.72,
        color=TOPK_SUBSTITUTED_COLOR,
        edgecolor=WHITE,
        linewidth=0.5,
    )

    for position, (k, row) in enumerate(substituted.iterrows()):
        ratio = float(row["ratio"])
        delta = float(substituted.loc[k, "substituted_delta"])
        label = f"{ratio * 100:.1f}%"
        if not np.isnan(delta):
            label += f"\n(+{delta * 100:.1f} %)"
        ax.text(
            position,
            ratio + 0.002,
            label,
            ha="center",
            va="bottom",
            fontsize=FONT_L,
            color=BLACK,
        )

    ax.set_xlabel(r"$k$", fontsize=FONT_LL, labelpad=PAD_S)
    ax.set_ylabel("Ratio of Substituted samples", fontsize=FONT_LL, labelpad=PAD_S)
    ax.set_ylim(0.55, 0.65)
    ax.set_xticks(x, [str(k) for k in substituted.index])
    ax.tick_params(axis="both", labelsize=FONT_L, pad=PAD_SS)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()

    plot_dir = ANALYSIS_RESULTS_DIR / "journal"
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        plot_dir / "mattergen_topk_substituted_ratios.pdf",
        bbox_inches="tight",
    )
    plt.show()


mattergen_topk_substituted_ratios = build_mattergen_topk_substituted_ratios(
    mattergen_classifications
)
display(mattergen_topk_substituted_ratios.round(3))
plot_mattergen_topk_substituted_ratios(mattergen_topk_substituted_ratios)

## MatterGen StructureMatcher threshold sensitivity

Novelty ratios for metastable SMACT-valid MatterGen samples using four `StructureMatcher` threshold levels. Candidate selection is fixed at the default setting; only direct and relaxed generated-to-training match checks vary.


In [ ]:
SM_THRESHOLD_SETTINGS = [
    {
        "label": "ltol=0.2, stol=0.3, angle_tol=5 (default)",
        "suffix": "",
        "ltol": 0.2,
        "stol": 0.3,
        "angle_tol": 5.0,
    },
    {
        "label": "ltol=0.15, stol=0.225, angle_tol=3.75",
        "suffix": "l0p15s0p225a3p75",
        "ltol": 0.15,
        "stol": 0.225,
        "angle_tol": 3.75,
    },
    {
        "label": "ltol=0.1, stol=0.15, angle_tol=2.5",
        "suffix": "l0p1s0p15a2p5",
        "ltol": 0.1,
        "stol": 0.15,
        "angle_tol": 2.5,
    },
    {
        "label": "ltol=0.05, stol=0.075, angle_tol=1.25",
        "suffix": "l0p05s0p075a1p25",
        "ltol": 0.05,
        "stol": 0.075,
        "angle_tol": 1.25,
    },
]
SM_THRESHOLD_LABELS = [setting["label"] for setting in SM_THRESHOLD_SETTINGS]
SM_SENSITIVITY_RESULT_DIR = RAW_RESULTS_DIR / "mattergen"


def sm_sensitivity_paths(suffix: str) -> dict[str, Path]:
    paths = model_required_paths("mattergen")
    suffix_part = f"_{suffix}" if suffix else ""
    paths["direct_sm"] = SM_SENSITIVITY_RESULT_DIR / f"sm_fit{suffix_part}.npz"
    paths["relaxed_sm_anon_matches"] = SM_SENSITIVITY_RESULT_DIR / (
        f"substituted_relaxed_niggli_top3_sm_anon_gen_matches{suffix_part}.pkl.gz"
    )
    paths["relaxed_wyckoff_matches"] = SM_SENSITIVITY_RESULT_DIR / (
        "substituted_relaxed_niggli_top3_wyckoff_match_s=0.01_"
        "c=mod_petti_gen_matches"
        f"{suffix_part}.pkl.gz"
    )
    return paths


def build_mattergen_sm_sensitivity_classifications() -> pd.DataFrame:
    frames = []
    selected_indices = None
    for setting in SM_THRESHOLD_SETTINGS:
        paths = sm_sensitivity_paths(setting["suffix"])
        missing = [path for path in paths.values() if not path.exists()]
        if missing:
            raise FileNotFoundError(
                "Missing sensitivity inputs:\n"
                + "\n".join(f"- {path}" for path in missing)
            )

        frame = classify_model(
            "mattergen",
            paths,
            include_category_label=True,
            include_crystal_system=True,
            include_simple_category=True,
            simple_category_labels=SIMPLE_CATEGORY_LABELS,
            simple_category_order=SIMPLE_CATEGORY_ORDER,
        )
        frame = frame[frame["is_metastable_smact_valid"]].copy()
        frame["threshold"] = setting["label"]
        current_indices = set(frame["gen_idx"].astype(int))
        if selected_indices is None:
            selected_indices = current_indices
        else:
            assert current_indices == selected_indices
        frames.append(frame)

    result = pd.concat(frames, ignore_index=True)
    result["threshold"] = pd.Categorical(
        result["threshold"],
        categories=SM_THRESHOLD_LABELS,
        ordered=True,
    )
    return result


def build_mattergen_sm_sensitivity_table(frame: pd.DataFrame) -> pd.DataFrame:
    counts = (
        frame.groupby(["threshold", "simple_category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [SM_THRESHOLD_LABELS, SIMPLE_CATEGORY_ORDER],
        names=["threshold", "simple_category"],
    )
    summary = counts.reindex(full_index, fill_value=0).reset_index()
    totals = summary.groupby("threshold", observed=False)["count"].transform("sum")
    summary["percent"] = 100 * summary["count"] / totals
    table = summary.pivot(
        index="threshold",
        columns="simple_category",
        values="percent",
    ).rename(columns=SIMPLE_CATEGORY_LABELS)
    table = table.rename(columns={label: f"{label} (%)" for label in table})
    table.insert(0, "n", totals.groupby(summary["threshold"]).first())

    parameters = pd.DataFrame(SM_THRESHOLD_SETTINGS).set_index("label")
    table.insert(1, "ltol", parameters["ltol"])
    table.insert(2, "stol", parameters["stol"])
    table.insert(3, "angle_tol", parameters["angle_tol"])
    table = table.reindex(SM_THRESHOLD_LABELS)
    table.index.name = "Threshold"
    table.columns.name = None
    percent_columns = [
        "Duplicate (%)",
        "Substituted (%)",
        "Unmatched (%)",
    ]
    assert table["n"].nunique() == 1
    assert np.allclose(table[percent_columns].sum(axis=1), 100.0)
    table[percent_columns] = table[percent_columns].round(2)
    return table


def plot_mattergen_sm_sensitivity_by_crystal_system(
    frame: pd.DataFrame,
) -> None:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(FIG_WIDTH, FIG_WIDTH * 0.85),
    )
    legend_handles = {}
    for index, (threshold, ax) in enumerate(
        zip(SM_THRESHOLD_LABELS, axes.ravel(), strict=True)
    ):
        selected = frame[frame["threshold"] == threshold]
        ratios = build_detailed_crystal_system_category_ratios(selected)
        handles = plot_detailed_category_ratios_by_crystal_system(
            ax,
            ratios,
            title=threshold,
            show_ylabel=index % 2 == 0,
        )
        if handles:
            legend_handles = handles

        ratio_sums = ratios.groupby("crystal_system", observed=False)["ratio"].sum()
        assert np.allclose(ratio_sums[ratio_sums > 0], 1.0)

    if legend_handles:
        fig.legend(
            handles=[legend_handles[category] for category in CATEGORY_ORDER],
            labels=[CATEGORY_LABELS[category] for category in CATEGORY_ORDER],
            loc="lower center",
            bbox_to_anchor=(0.5, -0.05),
            frameon=False,
            fontsize=FONT_M,
            ncol=len(CATEGORY_ORDER),
            handleheight=1.2,
            handlelength=1.5,
            handletextpad=0.5,
            columnspacing=1.0,
        )
    fig.tight_layout()

    plot_dir = ANALYSIS_RESULTS_DIR / "journal"
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        plot_dir / "mattergen_sm_threshold_category_ratios_by_crystal_system.pdf",
        bbox_inches="tight",
    )
    plt.show()


mattergen_sm_sensitivity = build_mattergen_sm_sensitivity_classifications()
mattergen_sm_sensitivity_table = build_mattergen_sm_sensitivity_table(
    mattergen_sm_sensitivity
)
display(mattergen_sm_sensitivity_table)
plot_mattergen_sm_sensitivity_by_crystal_system(mattergen_sm_sensitivity)

## MatterGen AMD UMAP train-density cross section

Train-fit UMAP scatterplot of metastable SMACT-valid MatterGen samples over
train-data density, with a KDE cross section at component 1 = 2.3.


In [ ]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

UMAP_MODEL = "mattergen"
UMAP_EMBEDDING = "amd"
UMAP_REDUCER = "umap"
UMAP_DISTANCE_METRIC = "chebyshev"
UMAP_FILTER_GENERATED_TO_METASTABLE_SMACT_VALID = True
UMAP_MAX_PLOT_POINTS_PER_GENERATED_CATEGORY = 50
UMAP_TRAIN_FIT_UMAP_SEED = 0
UMAP_TRAIN_FIT_SAMPLE_SEED = 0
UMAP_TRAIN_FIT_CROSS_SECTION_COMPONENT_1 = 2.3
UMAP_CROSS_SECTION_BAND_HALF_WIDTH = 0.5
UMAP_KDE_MIN_TRAIN_POINTS = 5
UMAP_KDE_SCOTT_FACTOR = 1.0
UMAP_GRID_SIZE = 140
UMAP_GENERATED_STRUCTURES_FILE = "relaxed_niggli.pkl.gz"
UMAP_TRAIN_STRUCTURES_FILE = "train.pkl.gz"
UMAP_TRAIN_FIT_CROSS_SECTION_CIF_DIR = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "mattergen_amd_train_fit_umap_cross_section_structures"
)
UMAP_TRAIN_FIT_CROSS_SECTION_NEAREST_TRAIN_TABLE_PATH = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "mattergen_amd_train_fit_umap_cross_section_nearest_train.csv"
)

UMAP_CATEGORY_LABELS = {
    "Duplicate": "Duplicate",
    "Substituted (both)": "Substituted",
    "Substituted (SM-anon only)": "Substituted",
    "Substituted (Wyckoff only)": "Substituted",
    "Substituted (Both)": "Substituted",
    "Substituted (Lattice & site only)": "Substituted",
    "Unmatched": "Unmatched",
    "exact match": "Duplicate",
    "subst match: both": "Substituted",
    "subst match: sm-anon": "Substituted",
    "subst match: wyckoff": "Substituted",
    "no match": "Unmatched",
}
UMAP_CATEGORY_ORDER = ["Duplicate", "Substituted", "Unmatched"]
UMAP_CATEGORY_COLORS = {
    "Duplicate": SIMPLE_CATEGORY_COLORS["1"],
    "Substituted": SIMPLE_CATEGORY_COLORS["2"],
    "Unmatched": SIMPLE_CATEGORY_COLORS["3"],
}
UMAP_CROSS_SECTION_TABLE_COLUMNS = [
    "gen_idx",
    "generated_formula",
    "simple_category_label",
    "crystal_system",
    "space_group",
    "umap_component_1",
    "umap_component_2",
    "nearest_train_idx",
    "nearest_train_formula",
    "amd_distance",
    "generated_cif_path",
]
MATPLOTLIB_CRYSTAL_SYSTEM_MARKERS = {
    "triclinic": "o",
    "monoclinic": "s",
    "orthorhombic": "D",
    "tetragonal": "P",
    "trigonal": "X",
    "hexagonal": "^",
    "cubic": "*",
}

AMD_EMBEDDING_SIZE = 100
AMD_TRAIN_STRUCTURES_PATH = INPUT_DIR / "train" / "preprocessed" / "train.pkl.gz"
AMD_PROJECTION_METADATA_COLUMNS = [
    "model",
    "split",
    "structure_idx",
    "gen_idx",
    "label",
    "composition",
    "crystal_system",
    "space_group",
    "is_metastable_smact_valid",
    "embedding_successful",
]
_AMD_TRAIN_EMBEDDINGS: np.ndarray | None = None
_AMD_GENERATED_INPUTS: dict[str, tuple[np.ndarray, pd.DataFrame]] = {}
_AMD_TRAIN_UMAP_FITS: dict[int, tuple[StandardScaler, Any, np.ndarray]] = {}


def load_embedding_projection_cache(path: Path) -> Any | None:
    if not path.exists():
        return None
    with gzip.open(path, "rb") as file:
        return pickle.load(file)  # noqa: S301


def save_embedding_projection_cache(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wb") as file:
        pickle.dump(payload, file, protocol=pickle.HIGHEST_PROTOCOL)


def amd_embedding_cache_path(
    *,
    model: str | None,
    n_samples: int,
) -> Path:
    prefix = "train" if model is None else model
    return (
        EMBEDDING_PROJECTION_CACHE_DIR
        / "embeddings"
        / f"{prefix}_amd_{n_samples}.pkl.gz"
    )


def compute_amd_embedding(
    structure: Any,
    *,
    structure_idx: int,
) -> np.ndarray:
    try:
        periodic_set = periodicset_from_pymatgen_structure(
            structure,
            remove_hydrogens=False,
            skip_disorder=False,
        )
        embedding = np.asarray(amd.AMD(periodic_set, AMD_EMBEDDING_SIZE), dtype=float)
    except Exception as exc:
        raise RuntimeError(f"AMD failed for structure {structure_idx}.") from exc

    expected_shape = (AMD_EMBEDDING_SIZE,)
    if embedding.shape != expected_shape:
        raise ValueError(
            f"AMD produced shape {embedding.shape} for structure {structure_idx}; "
            f"expected {expected_shape}."
        )
    if not np.all(np.isfinite(embedding)):
        raise ValueError(
            f"AMD produced non-finite values for structure {structure_idx}."
        )
    return embedding


def compute_amd_embedding_payload(
    structures: list[Any],
    *,
    description: str,
    allow_failures: bool,
) -> dict[str, Any]:
    embeddings: list[list[float] | None] = []
    successful_indices = []
    failed_indices = []
    failures = []
    for structure_idx, structure in enumerate(tqdm(structures, desc=description)):
        try:
            embedding = compute_amd_embedding(
                structure,
                structure_idx=structure_idx,
            )
        except Exception as exc:
            if not allow_failures:
                raise
            embeddings.append(None)
            failed_indices.append(structure_idx)
            failures.append(
                {
                    "structure_idx": structure_idx,
                    "exception_type": type(exc).__name__,
                    "message": str(exc),
                }
            )
            continue

        embeddings.append(embedding.tolist())
        successful_indices.append(structure_idx)

    return {
        "embeddings": embeddings,
        "successful_indices": successful_indices,
        "failed_indices": failed_indices,
        "failures": failures,
    }


def amd_embedding_payload_is_valid(
    payload: Any,
    *,
    model: str | None,
    n_samples: int,
) -> bool:
    if not isinstance(payload, dict):
        return False
    required = {
        "metadata",
        "embeddings",
        "successful_indices",
        "failed_indices",
        "failures",
    }
    if not required.issubset(payload):
        return False
    metadata = payload["metadata"]
    expected_metadata = {
        **({"model": model} if model is not None else {}),
        "split": "train" if model is None else "generated",
        "embedding_name": "amd",
        "n_samples": n_samples,
    }
    if any(metadata.get(key) != value for key, value in expected_metadata.items()):
        return False
    successful = [int(index) for index in payload["successful_indices"]]
    failed = [int(index) for index in payload["failed_indices"]]
    return (
        len(payload["embeddings"]) == n_samples
        and len(successful) + len(failed) == n_samples
        and sorted([*successful, *failed]) == list(range(n_samples))
    )


def load_or_compute_amd_embeddings(
    *,
    model: str | None,
    structures: list[Any],
    allow_failures: bool,
) -> dict[str, Any]:
    path = amd_embedding_cache_path(model=model, n_samples=len(structures))
    cached = load_embedding_projection_cache(path)
    if cached is not None and amd_embedding_payload_is_valid(
        cached,
        model=model,
        n_samples=len(structures),
    ):
        return cached

    split = "train" if model is None else "generated"
    description = f"{model or 'train'} AMD"
    payload = compute_amd_embedding_payload(
        structures,
        description=description,
        allow_failures=allow_failures,
    )
    payload = {
        "metadata": {
            **({"model": model} if model is not None else {}),
            "split": split,
            "embedding_name": "amd",
            "n_samples": len(structures),
        },
        "created_at": datetime.now(UTC).isoformat(),
        **payload,
    }
    save_embedding_projection_cache(path, payload)
    return payload


def successful_amd_embedding_array(payload: dict[str, Any]) -> np.ndarray:
    embeddings = payload["embeddings"]
    successful_indices = payload["successful_indices"]
    rows = [embeddings[index] for index in successful_indices]
    if not rows:
        return np.empty((0, AMD_EMBEDDING_SIZE), dtype=float)
    values = np.asarray(rows, dtype=float)
    if values.ndim != 2 or values.shape[1] != AMD_EMBEDDING_SIZE:
        raise ValueError(f"AMD embedding cache has unexpected shape {values.shape}.")
    if not np.all(np.isfinite(values)):
        raise ValueError("AMD embedding cache contains non-finite values.")
    return values


def amd_train_embeddings() -> np.ndarray:
    global _AMD_TRAIN_EMBEDDINGS
    if _AMD_TRAIN_EMBEDDINGS is None:
        structures = load_pickle_gz(AMD_TRAIN_STRUCTURES_PATH)
        payload = load_or_compute_amd_embeddings(
            model=None,
            structures=structures,
            allow_failures=False,
        )
        if payload["failed_indices"]:
            raise ValueError("Train AMD embedding cache contains failed structures.")
        _AMD_TRAIN_EMBEDDINGS = successful_amd_embedding_array(payload)
    return _AMD_TRAIN_EMBEDDINGS


def amd_generated_inputs(model: str) -> tuple[np.ndarray, pd.DataFrame]:
    cached = _AMD_GENERATED_INPUTS.get(model)
    if cached is not None:
        return cached
    if model not in complete_models:
        raise ValueError(
            f"Cannot compute AMD projection for incomplete or unknown model {model!r}."
        )

    paths = model_required_paths(model)
    structures = load_pickle_gz(paths["generated_structures"])
    payload = load_or_compute_amd_embeddings(
        model=model,
        structures=structures,
        allow_failures=True,
    )
    embeddings = successful_amd_embedding_array(payload)
    successful_indices = np.asarray(payload["successful_indices"], dtype=int)

    model_labels = (
        classifications[classifications["model"] == model]
        .sort_values("gen_idx")
        .reset_index(drop=True)
    )
    if len(model_labels) != len(structures):
        raise ValueError(
            f"{model} classifications contain {len(model_labels)} rows; "
            f"expected {len(structures)}."
        )
    wyckoff_data = load_pickle_gz(paths["wyckoff_repr"])
    if len(wyckoff_data) != len(structures):
        raise ValueError(
            f"{model} Wyckoff data contain {len(wyckoff_data)} rows; "
            f"expected {len(structures)}."
        )

    labels = model_labels.iloc[successful_indices].reset_index(drop=True)
    metadata = pd.DataFrame(
        {
            "model": model,
            "split": "generated",
            "structure_idx": labels["gen_idx"].to_numpy(),
            "gen_idx": labels["gen_idx"].to_numpy(),
            "label": labels["category_label"].astype(str),
            "composition": [
                structures[index].composition.reduced_formula
                for index in successful_indices
            ],
            "crystal_system": labels["crystal_system"].astype(str),
            "space_group": [
                int(wyckoff_data[index].spg_num) for index in successful_indices
            ],
            "is_metastable_smact_valid": labels["is_metastable_smact_valid"].astype(
                bool
            ),
            "embedding_successful": True,
        }
    )
    result = (embeddings, metadata)
    _AMD_GENERATED_INPUTS[model] = result
    return result


def amd_train_metadata(model: str, n_train: int) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "model": model,
            "split": "train",
            "structure_idx": np.arange(n_train),
            "gen_idx": pd.NA,
            "label": "train data",
            "composition": pd.NA,
            "crystal_system": pd.NA,
            "space_group": pd.NA,
            "is_metastable_smact_valid": pd.NA,
            "embedding_successful": True,
        }
    )


def selected_amd_generated_rows(
    metadata: pd.DataFrame,
    *,
    generated_cap: int | None,
    sample_seed: int,
) -> np.ndarray:
    generated = metadata[
        metadata["embedding_successful"].astype(bool)
        & metadata["is_metastable_smact_valid"].astype(bool)
    ].copy()
    if generated_cap is not None:
        generated["_sampling_label"] = (
            generated["label"].astype(str).map(UMAP_CATEGORY_LABELS)
        )
        if generated["_sampling_label"].isna().any():
            unknown = sorted(
                generated.loc[generated["_sampling_label"].isna(), "label"]
                .astype(str)
                .unique()
            )
            raise ValueError(f"Unknown AMD projection labels: {unknown}")
        sampled = [
            group
            if len(group) <= generated_cap
            else group.sample(
                n=generated_cap,
                random_state=sample_seed,
                replace=False,
            )
            for _, group in generated.groupby("_sampling_label", sort=True)
        ]
        generated = pd.concat(sampled) if sampled else generated.iloc[0:0]
    return np.asarray(sorted(generated.index), dtype=int)


def amd_train_umap_fit(
    umap_seed: int,
) -> tuple[StandardScaler, Any, np.ndarray]:
    cached = _AMD_TRAIN_UMAP_FITS.get(umap_seed)
    if cached is not None:
        return cached
    train_embeddings = amd_train_embeddings()
    scaler = StandardScaler().fit(train_embeddings)
    train_scaled = scaler.transform(train_embeddings)
    reducer = umap.UMAP(
        n_components=2,
        random_state=umap_seed,
        metric=UMAP_DISTANCE_METRIC,
    ).fit(train_scaled)
    result = (scaler, reducer, reducer.transform(train_scaled))
    _AMD_TRAIN_UMAP_FITS[umap_seed] = result
    return result


def amd_umap_projection_cache_path(
    model: str,
    *,
    selection: str,
    generated_cap: int | None,
    umap_seed: int,
    sample_seed: int = 0,
) -> Path:
    cap_label = "all" if generated_cap is None else str(generated_cap)
    sample_label = "" if generated_cap is None else f"_sample_seed={sample_seed}"
    name = (
        f"train_fit_{model}_amd_umap_{selection}"
        f"_metric={UMAP_DISTANCE_METRIC}"
        f"_f={UMAP_FILTER_GENERATED_TO_METASTABLE_SMACT_VALID}"
        f"_n={cap_label}"
        f"_umap_seed={umap_seed}"
        f"{sample_label}.pkl.gz"
    )
    return EMBEDDING_PROJECTION_CACHE_DIR / "projections" / name


def projection_cache_is_valid(
    cached: Any,
    expected_metadata: pd.DataFrame,
) -> bool:
    if not isinstance(cached, pd.DataFrame):
        return False
    required = {*AMD_PROJECTION_METADATA_COLUMNS, "x", "y", "reducer"}
    if not required.issubset(cached.columns) or len(cached) != len(expected_metadata):
        return False
    if not (cached["reducer"] == UMAP_REDUCER).all():
        return False
    if not np.all(np.isfinite(cached[["x", "y"]].to_numpy(dtype=float))):
        return False
    try:
        pd.testing.assert_frame_equal(
            cached[AMD_PROJECTION_METADATA_COLUMNS].reset_index(drop=True),
            expected_metadata[AMD_PROJECTION_METADATA_COLUMNS].reset_index(drop=True),
            check_dtype=False,
            check_categorical=False,
        )
    except AssertionError:
        return False
    return True


def load_or_compute_train_fit_amd_umap_projection(
    *,
    model: str,
    path: Path,
    generated_cap: int | None,
    umap_seed: int,
    sample_seed: int,
) -> pd.DataFrame:
    train_embeddings = amd_train_embeddings()
    generated_embeddings, generated_metadata = amd_generated_inputs(model)
    selected_rows = selected_amd_generated_rows(
        generated_metadata,
        generated_cap=generated_cap,
        sample_seed=sample_seed,
    )
    expected_metadata = pd.concat(
        [
            amd_train_metadata(model, len(train_embeddings)),
            generated_metadata.iloc[selected_rows].reset_index(drop=True),
        ],
        ignore_index=True,
    )
    cached = load_embedding_projection_cache(path)
    if projection_cache_is_valid(cached, expected_metadata):
        return cached.copy()

    scaler, reducer, train_coords = amd_train_umap_fit(umap_seed)
    projection_parts = [
        expected_metadata.iloc[: len(train_embeddings)].assign(
            x=train_coords[:, 0],
            y=train_coords[:, 1],
            reducer=UMAP_REDUCER,
        )
    ]
    if len(selected_rows):
        generated_coords = reducer.transform(
            scaler.transform(generated_embeddings[selected_rows])
        )
        projection_parts.append(
            expected_metadata.iloc[len(train_embeddings) :].assign(
                x=generated_coords[:, 0],
                y=generated_coords[:, 1],
                reducer=UMAP_REDUCER,
            )
        )
    frame = pd.concat(projection_parts, ignore_index=True)
    save_embedding_projection_cache(path, frame)
    return frame.copy()


def load_mattergen_train_fit_umap_projection(
    *,
    umap_seed: int = UMAP_TRAIN_FIT_UMAP_SEED,
    sample_seed: int = UMAP_TRAIN_FIT_SAMPLE_SEED,
) -> pd.DataFrame:
    path = amd_umap_projection_cache_path(
        UMAP_MODEL,
        selection="scatter",
        generated_cap=UMAP_MAX_PLOT_POINTS_PER_GENERATED_CATEGORY,
        umap_seed=umap_seed,
        sample_seed=sample_seed,
    )
    frame = load_or_compute_train_fit_amd_umap_projection(
        model=UMAP_MODEL,
        path=path,
        generated_cap=UMAP_MAX_PLOT_POINTS_PER_GENERATED_CATEGORY,
        umap_seed=umap_seed,
        sample_seed=sample_seed,
    )
    required_columns = {"split", "label", "crystal_system", "x", "y"}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"Projection cache is missing columns: {missing_columns}")
    return frame.copy()


def compute_amd_distances_to_train(
    generated_embeddings: np.ndarray,
    train_embeddings: np.ndarray,
) -> np.ndarray:
    chunks = [
        np.asarray(chunk, dtype=np.float32)
        for chunk in pairwise_distances_chunked(
            generated_embeddings,
            train_embeddings,
            metric=UMAP_DISTANCE_METRIC,
        )
    ]
    return np.vstack(chunks)


def mattergen_nearest_train_rows(gen_indices: list[int]) -> pd.DataFrame:
    generated_embeddings, metadata = amd_generated_inputs(UMAP_MODEL)
    row_by_gen_idx = pd.Series(
        np.arange(len(metadata), dtype=int),
        index=metadata["gen_idx"].astype(int),
    )
    missing = sorted(set(gen_indices) - set(row_by_gen_idx.index))
    if missing:
        raise ValueError(
            f"MatterGen AMD embeddings are unavailable for gen_idx values {missing}."
        )
    rows = row_by_gen_idx.loc[gen_indices].to_numpy(dtype=int)
    distances = compute_amd_distances_to_train(
        generated_embeddings[rows],
        amd_train_embeddings(),
    )
    nearest_indices = distances.argmin(axis=1)
    return pd.DataFrame(
        {
            "gen_idx": gen_indices,
            "nearest_train_idx": nearest_indices.astype(int),
            "amd_distance": distances[
                np.arange(len(distances)),
                nearest_indices,
            ],
        }
    ).set_index("gen_idx")


def add_umap_simple_category(
    frame: pd.DataFrame,
    *,
    model: str | None = None,
) -> pd.DataFrame:
    mapped = frame["label"].astype(str).map(UMAP_CATEGORY_LABELS)
    generated = frame["split"] == "generated"
    unknown = sorted(frame.loc[generated & mapped.isna(), "label"].astype(str).unique())
    if unknown:
        label = f" for {model}" if model is not None else ""
        raise ValueError(f"Unknown category labels{label}: {unknown}")
    return frame.assign(simple_category_label=mapped)


def scott_kde_bandwidth(values: np.ndarray, factor: float) -> float:
    dimension = values.shape[1]
    coordinate_scale = float(np.mean(np.std(values, axis=0, ddof=1)))
    if not np.isfinite(coordinate_scale) or coordinate_scale <= 0:
        coordinate_scale = 1.0
    return factor * coordinate_scale * len(values) ** (-1 / (dimension + 4))


def fit_train_projection_kde(
    frame: pd.DataFrame,
    *,
    factor: float = UMAP_KDE_SCOTT_FACTOR,
) -> KernelDensity:
    train_xy = frame.loc[frame["split"] == "train", ["x", "y"]].to_numpy(dtype=float)
    train_xy = train_xy[np.isfinite(train_xy).all(axis=1)]
    if len(train_xy) < UMAP_KDE_MIN_TRAIN_POINTS:
        raise ValueError("Too few finite train projection points for KDE.")
    bandwidth = scott_kde_bandwidth(train_xy, factor)
    return KernelDensity(kernel="gaussian", bandwidth=bandwidth).fit(train_xy)


def projection_limits(values: pd.Series) -> tuple[float, float]:
    lower = float(np.nanmin(values.astype(float)))
    upper = float(np.nanmax(values.astype(float)))
    span = float(upper - lower)
    if not np.isfinite(span) or span <= 0:
        return lower - 1.0, upper + 1.0
    pad = 0.04 * span
    return lower - pad, upper + pad


def kde_grid(
    kde: KernelDensity,
    *,
    x_limits: tuple[float, float],
    y_limits: tuple[float, float],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x_values = np.linspace(*x_limits, UMAP_GRID_SIZE)
    y_values = np.linspace(*y_limits, UMAP_GRID_SIZE)
    xx, yy = np.meshgrid(x_values, y_values)
    points = np.column_stack([xx.ravel(), yy.ravel()])
    density = np.exp(kde.score_samples(points)).reshape(xx.shape)
    return xx, yy, density


def kde_cross_section(
    kde: KernelDensity,
    *,
    component_1: float,
    y_limits: tuple[float, float],
) -> tuple[np.ndarray, np.ndarray]:
    y_values = np.linspace(*y_limits, UMAP_GRID_SIZE * 2)
    points = np.column_stack([np.full_like(y_values, component_1), y_values])
    return y_values, np.exp(kde.score_samples(points))


def add_umap_scatter(ax: plt.Axes, generated: pd.DataFrame) -> None:
    for category in UMAP_CATEGORY_ORDER:
        category_frame = generated[generated["simple_category_label"] == category]
        if category_frame.empty:
            continue
        for crystal_system in CRYSTAL_SYSTEM_ORDER:
            subset = category_frame[category_frame["crystal_system"] == crystal_system]
            if subset.empty:
                continue
            ax.scatter(
                subset["x"],
                subset["y"],
                s=32,
                c=UMAP_CATEGORY_COLORS[category],
                marker=MATPLOTLIB_CRYSTAL_SYSTEM_MARKERS[crystal_system],
                alpha=0.65,
                edgecolors="none",
            )


def umap_category_handles() -> list[Any]:
    return [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=UMAP_CATEGORY_COLORS[category],
            markeredgecolor="none",
            markersize=7,
            label=category,
        )
        for category in UMAP_CATEGORY_ORDER
    ]


def crystal_system_handles() -> list[Any]:
    return [
        plt.Line2D(
            [0],
            [0],
            marker=MATPLOTLIB_CRYSTAL_SYSTEM_MARKERS[crystal_system],
            color="none",
            markerfacecolor=BLACK,
            markeredgecolor="none",
            markersize=7,
            label=crystal_system,
        )
        for crystal_system in CRYSTAL_SYSTEM_ORDER
    ]


def projection_limits_for_frames(
    frames: dict[Any, pd.DataFrame],
    column: str,
) -> tuple[float, float]:
    values = pd.concat([frame[column].astype(float) for frame in frames.values()])
    values = values[np.isfinite(values)]
    if values.empty:
        raise ValueError(f"No finite {column!r} projection values available.")
    return projection_limits(values)


def train_projection_density(
    frame: pd.DataFrame,
    *,
    x_limits: tuple[float, float],
    y_limits: tuple[float, float],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    return kde_grid(
        fit_train_projection_kde(frame),
        x_limits=x_limits,
        y_limits=y_limits,
    )


def add_train_projection_density(
    ax: plt.Axes,
    frame: pd.DataFrame,
    *,
    x_limits: tuple[float, float],
    y_limits: tuple[float, float],
    density: tuple[np.ndarray, np.ndarray, np.ndarray] | None = None,
) -> None:
    xx, yy, values = density or train_projection_density(
        frame,
        x_limits=x_limits,
        y_limits=y_limits,
    )
    positive = values[values > 0]
    if len(positive) == 0:
        return
    levels = np.linspace(float(positive.min()), float(values.max()), 10)
    ax.contourf(xx, yy, values, levels=levels, cmap="Greys", alpha=0.42)
    ax.contour(xx, yy, values, levels=levels, colors=GRAY, linewidths=0.6)


def generated_metastable_smact_valid_projection(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    generated = frame[frame["split"] == "generated"].copy()
    generated = generated[
        generated["embedding_successful"].astype(bool)
        & generated["is_metastable_smact_valid"].astype(bool)
    ].copy()
    generated_xy = generated[["x", "y"]].to_numpy(dtype=float)
    return generated.loc[np.isfinite(generated_xy).all(axis=1)].copy()


def mattergen_umap_cross_section_band(
    generated: pd.DataFrame,
    *,
    component_1: float,
) -> pd.DataFrame:
    return (
        generated[
            generated["x"].between(
                component_1 - UMAP_CROSS_SECTION_BAND_HALF_WIDTH,
                component_1 + UMAP_CROSS_SECTION_BAND_HALF_WIDTH,
            )
        ]
        .sort_values("y")
        .copy()
    )


def add_umap_cross_section_markers(
    ax: plt.Axes,
    generated: pd.DataFrame,
    kde: KernelDensity,
    *,
    component_1: float,
) -> None:
    band = mattergen_umap_cross_section_band(generated, component_1=component_1)
    if band.empty:
        return

    points = np.column_stack([np.full(len(band), component_1), band["y"].to_numpy()])
    band["density"] = np.exp(kde.score_samples(points))
    for category in UMAP_CATEGORY_ORDER:
        category_frame = band[band["simple_category_label"] == category]
        if category_frame.empty:
            continue
        for crystal_system in CRYSTAL_SYSTEM_ORDER:
            subset = category_frame[category_frame["crystal_system"] == crystal_system]
            if subset.empty:
                continue
            ax.scatter(
                subset["y"],
                subset["density"],
                s=34,
                c=UMAP_CATEGORY_COLORS[category],
                marker=MATPLOTLIB_CRYSTAL_SYSTEM_MARKERS[crystal_system],
                alpha=0.65,
                edgecolors="none",
            )


def safe_filename_label(value: object) -> str:
    label = "".join(char if char.isalnum() else "_" for char in str(value))
    return label.strip("_") or "unknown"


def conventional_structure(structure: object) -> object:
    return SpacegroupAnalyzer(structure).get_conventional_standard_structure()


def mattergen_cross_section_generated_cif_path(
    gen_idx: int,
    structure: object,
    *,
    cif_dir: Path,
) -> Path:
    formula = safe_filename_label(structure.composition.reduced_formula)
    return cif_dir / (
        f"{UMAP_MODEL}_{UMAP_EMBEDDING}_{UMAP_REDUCER}_gen_{gen_idx}_{formula}.cif"
    )


def save_mattergen_umap_cross_section_structures_and_table(
    generated: pd.DataFrame,
    *,
    component_1: float = UMAP_TRAIN_FIT_CROSS_SECTION_COMPONENT_1,
    cif_dir: Path = UMAP_TRAIN_FIT_CROSS_SECTION_CIF_DIR,
    nearest_train_table_path: Path = (
        UMAP_TRAIN_FIT_CROSS_SECTION_NEAREST_TRAIN_TABLE_PATH
    ),
) -> pd.DataFrame:
    band = mattergen_umap_cross_section_band(generated, component_1=component_1)
    generated_structures = load_pickle_gz(
        INPUT_DIR / "gen" / "preprocessed" / UMAP_MODEL / UMAP_GENERATED_STRUCTURES_FILE
    )
    train_structures = load_pickle_gz(
        INPUT_DIR / "train" / "preprocessed" / UMAP_TRAIN_STRUCTURES_FILE
    )
    nearest_rows = mattergen_nearest_train_rows(
        [int(gen_idx) for gen_idx in band["gen_idx"]]
    )

    cif_dir.mkdir(parents=True, exist_ok=True)
    records = []
    for row in band.itertuples(index=False):
        gen_idx = int(row.gen_idx)
        generated_structure = conventional_structure(generated_structures[gen_idx])
        nearest_train_idx = int(nearest_rows.loc[gen_idx, "nearest_train_idx"])
        nearest_train_structure = train_structures[nearest_train_idx]
        cif_path = mattergen_cross_section_generated_cif_path(
            gen_idx,
            generated_structure,
            cif_dir=cif_dir,
        )
        generated_structure.to(filename=cif_path)
        records.append(
            {
                "gen_idx": gen_idx,
                "generated_formula": generated_structure.composition.reduced_formula,
                "simple_category_label": row.simple_category_label,
                "crystal_system": row.crystal_system,
                "space_group": int(row.space_group),
                "umap_component_1": float(row.x),
                "umap_component_2": float(row.y),
                "nearest_train_idx": nearest_train_idx,
                "nearest_train_formula": (
                    nearest_train_structure.composition.reduced_formula
                ),
                "amd_distance": float(nearest_rows.loc[gen_idx, "amd_distance"]),
                "generated_cif_path": str(cif_path.relative_to(ROOT)),
            }
        )

    table = pd.DataFrame.from_records(records, columns=UMAP_CROSS_SECTION_TABLE_COLUMNS)
    nearest_train_table_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    table.to_csv(nearest_train_table_path, index=False)
    return table


def plot_mattergen_umap_density_cross_section(
    path: Path,
    *,
    projection_loader: Any = load_mattergen_train_fit_umap_projection,
    component_1: float = UMAP_TRAIN_FIT_CROSS_SECTION_COMPONENT_1,
    cif_dir: Path = UMAP_TRAIN_FIT_CROSS_SECTION_CIF_DIR,
    nearest_train_table_path: Path = (
        UMAP_TRAIN_FIT_CROSS_SECTION_NEAREST_TRAIN_TABLE_PATH
    ),
) -> None:
    projections = add_umap_simple_category(projection_loader())
    frame = projections[projections["reducer"] == UMAP_REDUCER].copy()
    generated = frame[
        (frame["split"] == "generated") & frame["simple_category_label"].notna()
    ].copy()
    if frame.empty or generated.empty:
        display(Markdown("No MatterGen AMD UMAP projections available."))
        return

    x_limits = projection_limits(frame["x"])
    y_limits = projection_limits(frame["y"])
    kde = fit_train_projection_kde(frame)
    xx, yy, density = kde_grid(kde, x_limits=x_limits, y_limits=y_limits)
    section_y, section_density = kde_cross_section(
        kde,
        component_1=component_1,
        y_limits=y_limits,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(FIG_WIDTH, FIG_WIDTH * 0.5),
        constrained_layout=True,
    )
    positive_density = density[density > 0]
    if len(positive_density):
        levels = np.linspace(float(positive_density.min()), float(density.max()), 10)
        axes[0].contourf(xx, yy, density, levels=levels, cmap="Greys", alpha=0.42)
        axes[0].contour(xx, yy, density, levels=levels, colors=GRAY, linewidths=0.6)
    add_umap_scatter(axes[0], generated)
    axes[0].axvline(
        component_1,
        color=BLACK,
        linestyle="--",
        linewidth=1.0,
        alpha=0.8,
    )
    axes[0].set_xlabel("UMAP 1", fontsize=FONT_LL)
    axes[0].set_ylabel("UMAP 2", fontsize=FONT_LL)
    axes[0].set_xlim(x_limits)
    axes[0].set_ylim(y_limits)

    axes[1].plot(section_y, section_density, color=GRAY, linewidth=1.0)
    add_umap_cross_section_markers(
        axes[1],
        generated,
        kde,
        component_1=component_1,
    )
    axes[1].set_xlabel("UMAP 2", fontsize=FONT_LL)
    axes[1].set_ylabel("Train KDE density", fontsize=FONT_LL)
    axes[1].set_xlim(y_limits)
    axes[1].set_ylim(bottom=0.0)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(axis="both", labelsize=FONT_M, pad=PAD_SS)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for tag, ax in zip(("(a)", "(b)"), axes, strict=True):
        ax.text(
            -0.08,
            1.07,
            tag,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=FONT_LL,
            fontweight="bold",
            color=BLACK,
        )

    category_handles = umap_category_handles()
    crystal_handles = crystal_system_handles()
    fig.legend(
        category_handles,
        [handle.get_label() for handle in category_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.17),
        ncol=len(category_handles),
        frameon=False,
        fontsize=FONT_L,
        handletextpad=0,
        columnspacing=1.0,
    )
    fig.legend(
        crystal_handles,
        [handle.get_label() for handle in crystal_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.1),
        ncol=len(crystal_handles),
        frameon=False,
        fontsize=FONT_L,
        handletextpad=0,
        columnspacing=1.0,
    )

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", dpi=300)
    plt.show()
    display(
        save_mattergen_umap_cross_section_structures_and_table(
            generated,
            component_1=component_1,
            cif_dir=cif_dir,
            nearest_train_table_path=nearest_train_table_path,
        )
    )


plot_mattergen_umap_density_cross_section(
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "mattergen_amd_train_fit_umap_train_density_cross_section.png",
    projection_loader=load_mattergen_train_fit_umap_projection,
    component_1=UMAP_TRAIN_FIT_CROSS_SECTION_COMPONENT_1,
    cif_dir=UMAP_TRAIN_FIT_CROSS_SECTION_CIF_DIR,
    nearest_train_table_path=UMAP_TRAIN_FIT_CROSS_SECTION_NEAREST_TRAIN_TABLE_PATH,
)

## MatterGen AMD UMAP seed sensitivity

Train-fit AMD UMAP scatterplots across UMAP and generated-sample random seeds.

In [ ]:
MATTERGEN_UMAP_SEED_VALUES = (0, 1, 2)
MATTERGEN_SAMPLE_SEED_VALUES = (0, 1, 2)
MATTERGEN_UMAP_SEED_COMPARISON_OUTPUT_PATH = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "mattergen_amd_train_fit_umap_umap_seed_comparison.pdf"
)
MATTERGEN_SAMPLE_SEED_COMPARISON_OUTPUT_PATH = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "mattergen_amd_train_fit_umap_sample_seed_comparison.pdf"
)


def mattergen_train_fit_umap_seed_frame(
    *,
    umap_seed: int,
    sample_seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    frame = add_umap_simple_category(
        load_mattergen_train_fit_umap_projection(
            umap_seed=umap_seed,
            sample_seed=sample_seed,
        )
    )
    frame = frame[frame["reducer"] == UMAP_REDUCER].copy()
    generated = frame[
        (frame["split"] == "generated") & frame["simple_category_label"].notna()
    ].copy()
    if frame.empty or generated.empty:
        raise ValueError(
            "No MatterGen AMD UMAP projections available for "
            f"umap_seed={umap_seed}, sample_seed={sample_seed}."
        )
    return frame, generated


def plot_mattergen_amd_umap_seed_grid(
    path: Path,
    *,
    varied_seed: str,
    seed_values: tuple[int, int, int],
) -> None:
    if varied_seed not in {"umap", "sample"}:
        raise ValueError("varied_seed must be 'umap' or 'sample'.")

    frames = {}
    generated_frames = {}
    for seed in seed_values:
        umap_seed = seed if varied_seed == "umap" else UMAP_TRAIN_FIT_UMAP_SEED
        sample_seed = seed if varied_seed == "sample" else UMAP_TRAIN_FIT_SAMPLE_SEED
        frame, generated = mattergen_train_fit_umap_seed_frame(
            umap_seed=umap_seed,
            sample_seed=sample_seed,
        )
        frames[seed] = frame
        generated_frames[seed] = generated

    shared_limits = varied_seed == "sample"
    shared_density = None
    if shared_limits:
        shared_x_limits = projection_limits_for_frames(frames, "x")
        shared_y_limits = projection_limits_for_frames(frames, "y")
        shared_density = train_projection_density(
            next(iter(frames.values())),
            x_limits=shared_x_limits,
            y_limits=shared_y_limits,
        )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(FIG_WIDTH, FIG_WIDTH * 0.35),
        constrained_layout=True,
    )
    seed_title = "UMAP seed" if varied_seed == "umap" else "Sample seed"
    for ax, seed in zip(axes, seed_values, strict=True):
        frame = frames[seed]
        x_limits = shared_x_limits if shared_limits else projection_limits(frame["x"])
        y_limits = shared_y_limits if shared_limits else projection_limits(frame["y"])
        add_train_projection_density(
            ax,
            frame,
            x_limits=x_limits,
            y_limits=y_limits,
            density=shared_density,
        )
        add_umap_scatter(ax, generated_frames[seed])
        ax.set_title(f"{seed_title} = {seed}", fontsize=FONT_LL)
        ax.set_xlabel("UMAP 1", fontsize=FONT_L, labelpad=PAD_S)
        ax.set_ylabel("UMAP 2", fontsize=FONT_L, labelpad=PAD_S)
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    category_handles = umap_category_handles()
    crystal_handles = crystal_system_handles()
    fig.legend(
        category_handles,
        [handle.get_label() for handle in category_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.25),
        ncol=len(category_handles),
        frameon=False,
        fontsize=FONT_L,
        handletextpad=0,
        columnspacing=1.0,
    )
    fig.legend(
        crystal_handles,
        [handle.get_label() for handle in crystal_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        ncol=len(crystal_handles),
        frameon=False,
        fontsize=FONT_L,
        handletextpad=0,
        columnspacing=0.8,
    )

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", dpi=300)
    plt.show()


plot_mattergen_amd_umap_seed_grid(
    MATTERGEN_UMAP_SEED_COMPARISON_OUTPUT_PATH,
    varied_seed="umap",
    seed_values=MATTERGEN_UMAP_SEED_VALUES,
)
plot_mattergen_amd_umap_seed_grid(
    MATTERGEN_SAMPLE_SEED_COMPARISON_OUTPUT_PATH,
    varied_seed="sample",
    seed_values=MATTERGEN_SAMPLE_SEED_VALUES,
)

## AMD UMAP projected train-density likelihood

Class-wise density of train-fit AMD UMAP train-density log-likelihood for filtered MatterGen and MP20 Test samples.

In [ ]:
AMD_UMAP_LIKELIHOOD_MODELS = ("mattergen", "test")
AMD_UMAP_LIKELIHOOD_KDE_SCOTT_FACTOR = 0.5
AMD_UMAP_LIKELIHOOD_GRID_SIZE = 512
AMD_UMAP_LIKELIHOOD_X_MIN = -7
AMD_UMAP_LIKELIHOOD_X_MAX = -4.5
AMD_UMAP_LIKELIHOOD_OUTPUT_PATH = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "amd_umap_projected_train_density_likelihood_mattergen_test.pdf"
)


def load_train_fit_amd_umap_likelihood_projection(model: str) -> pd.DataFrame:
    path = amd_umap_projection_cache_path(
        model,
        selection="likelihood_all_filtered",
        generated_cap=None,
        umap_seed=0,
    )
    frame = load_or_compute_train_fit_amd_umap_projection(
        model=model,
        path=path,
        generated_cap=None,
        umap_seed=0,
        sample_seed=0,
    )

    required_columns = {
        "split",
        "label",
        "embedding_successful",
        "is_metastable_smact_valid",
        "x",
        "y",
        "reducer",
    }
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{path} is missing columns: {missing_columns}")
    return frame.copy()


def train_fit_amd_umap_likelihood_frame(model: str) -> pd.DataFrame:
    frame = load_train_fit_amd_umap_likelihood_projection(model)
    frame = frame[frame["reducer"] == UMAP_REDUCER].copy()
    kde = fit_train_projection_kde(
        frame,
        factor=AMD_UMAP_LIKELIHOOD_KDE_SCOTT_FACTOR,
    )

    generated = frame[frame["split"] == "generated"].copy()
    generated = generated[
        generated["embedding_successful"].astype(bool)
        & generated["is_metastable_smact_valid"].astype(bool)
    ].copy()
    generated["simple_category_label"] = (
        generated["label"].astype(str).map(UMAP_CATEGORY_LABELS)
    )
    if generated["simple_category_label"].isna().any():
        unknown_labels = sorted(
            generated.loc[generated["simple_category_label"].isna(), "label"]
            .astype(str)
            .unique()
        )
        raise ValueError(f"Unknown category labels for {model}: {unknown_labels}")

    generated_xy = generated[["x", "y"]].to_numpy(dtype=float)
    finite_mask = np.isfinite(generated_xy).all(axis=1)
    generated = generated.loc[finite_mask].copy()
    generated_xy = generated_xy[finite_mask]
    generated["train_kde_log_likelihood"] = kde.score_samples(generated_xy)
    return generated


def amd_umap_likelihood_kde_density(
    values: pd.Series,
    x_grid: np.ndarray,
) -> np.ndarray | None:
    finite_values = values.to_numpy(dtype=float)
    finite_values = finite_values[np.isfinite(finite_values)]
    if len(finite_values) < 2 or len(np.unique(finite_values)) < 2:
        return None

    bandwidth = scott_kde_bandwidth(
        finite_values[:, None],
        AMD_UMAP_LIKELIHOOD_KDE_SCOTT_FACTOR,
    )
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth).fit(
        finite_values[:, None]
    )
    return np.exp(kde.score_samples(x_grid[:, None]))


def amd_umap_likelihood_bounds(
    frames: dict[str, pd.DataFrame],
) -> tuple[float, float]:
    values = pd.concat(
        [frame["train_kde_log_likelihood"] for frame in frames.values()],
        ignore_index=True,
    )
    finite = values[np.isfinite(values)]
    if finite.empty:
        raise ValueError("No finite train KDE log-likelihood values available.")
    lower = (
        float(finite.min())
        if AMD_UMAP_LIKELIHOOD_X_MIN is None
        else AMD_UMAP_LIKELIHOOD_X_MIN
    )
    upper = (
        float(finite.max())
        if AMD_UMAP_LIKELIHOOD_X_MAX is None
        else AMD_UMAP_LIKELIHOOD_X_MAX
    )
    if lower == upper:
        padding = max(abs(lower) * 0.01, 1.0)
        return lower - padding, upper + padding
    return lower, upper


def add_amd_umap_likelihood_panel(
    ax: plt.Axes,
    frame: pd.DataFrame,
    x_grid: np.ndarray,
) -> None:
    total_count = len(frame)
    if total_count == 0:
        ax.set_visible(False)
        return
    total_scaled_density = np.zeros_like(x_grid)
    for category in UMAP_CATEGORY_ORDER:
        values = frame.loc[
            frame["simple_category_label"] == category,
            "train_kde_log_likelihood",
        ]
        density = amd_umap_likelihood_kde_density(values, x_grid)
        if density is None:
            continue
        scaled_density = density * len(values) / total_count
        total_scaled_density += scaled_density
        color = UMAP_CATEGORY_COLORS[category]
        ax.fill_between(x_grid, scaled_density, color=color, alpha=0.25)
        ax.plot(x_grid, scaled_density, color=color, linewidth=1.8)
    ax.plot(x_grid, total_scaled_density, color=GRAY, linewidth=2.2)


def amd_umap_likelihood_legend_handles() -> list[Any]:
    return [
        *[
            plt.Line2D(
                [0],
                [0],
                color=UMAP_CATEGORY_COLORS[category],
                linewidth=1.8,
                label=category,
            )
            for category in UMAP_CATEGORY_ORDER
        ],
        plt.Line2D([0], [0], color=GRAY, linewidth=2.2, label="Total"),
    ]


def plot_amd_umap_train_density_likelihood() -> None:
    frames = {
        model: train_fit_amd_umap_likelihood_frame(model)
        for model in AMD_UMAP_LIKELIHOOD_MODELS
    }
    lower, upper = amd_umap_likelihood_bounds(frames)
    x_grid = np.linspace(lower, upper, AMD_UMAP_LIKELIHOOD_GRID_SIZE)

    fig, axes = plt.subplots(
        1,
        len(AMD_UMAP_LIKELIHOOD_MODELS),
        figsize=(FIG_WIDTH, FIG_WIDTH * 0.5),
        sharex=True,
        sharey=True,
    )
    for ax, model in zip(np.ravel(axes), AMD_UMAP_LIKELIHOOD_MODELS, strict=True):
        frame = frames[model]
        add_amd_umap_likelihood_panel(ax, frame, x_grid)

        ax.set_title(MODEL_DISPLAY_NAMES.get(model, model), fontsize=FONT_LL)
        ax.set_xlabel("Train KDE log-likelihood", fontsize=FONT_L, labelpad=PAD_S)
        ax.tick_params(axis="both", labelsize=FONT_M, pad=PAD_SS)
        ax.tick_params(axis="y", labelleft=True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    axes[0].set_ylabel("Density", fontsize=FONT_L, labelpad=PAD_S)

    for tag, ax in zip(("(a)", "(b)"), np.ravel(axes), strict=True):
        ax.text(
            -0.20,
            1.15,
            tag,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=FONT_LL,
            fontweight="bold",
            color=BLACK,
        )

    legend_handles = amd_umap_likelihood_legend_handles()
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.0),
        frameon=False,
        fontsize=FONT_L,
        ncol=len(legend_handles),
    )
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))

    AMD_UMAP_LIKELIHOOD_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(AMD_UMAP_LIKELIHOOD_OUTPUT_PATH, bbox_inches="tight")
    plt.show()


plot_amd_umap_train_density_likelihood()

## AMD UMAP scatter by generated model

Train-fit AMD UMAP scatterplots for metastable SMACT-valid samples from DiffCSP++, WyckoffTransformer, Crystalite, and Chemeleon2.


In [ ]:
MODEL_COMPARISON_UMAP_MODELS = (
    "diffcsppp",
    "wyckofftransformer",
    "crystalite",
    "chemeleon2",
)
MODEL_COMPARISON_UMAP_MAX_PLOT_POINTS_PER_GENERATED_CATEGORY = 50
MODEL_COMPARISON_UMAP_SCATTER_OUTPUT_PATH = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "amd_umap_metastable_smact_valid_model_scatter_2x2.pdf"
)


def load_train_fit_amd_umap_scatter_projection(model: str) -> pd.DataFrame:
    path = amd_umap_projection_cache_path(
        model,
        selection="scatter",
        generated_cap=MODEL_COMPARISON_UMAP_MAX_PLOT_POINTS_PER_GENERATED_CATEGORY,
        umap_seed=0,
        sample_seed=0,
    )
    frame = load_or_compute_train_fit_amd_umap_projection(
        model=model,
        path=path,
        generated_cap=MODEL_COMPARISON_UMAP_MAX_PLOT_POINTS_PER_GENERATED_CATEGORY,
        umap_seed=0,
        sample_seed=0,
    )

    required_columns = {
        "split",
        "label",
        "embedding_successful",
        "is_metastable_smact_valid",
        "crystal_system",
        "x",
        "y",
        "reducer",
    }
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{path} is missing columns: {missing_columns}")
    return frame.copy()


def train_fit_amd_umap_scatter_frame(model: str) -> pd.DataFrame:
    frame = load_train_fit_amd_umap_scatter_projection(model)
    frame = frame[frame["reducer"] == UMAP_REDUCER].copy()
    frame["simple_category_label"] = (
        frame["label"].astype(str).map(UMAP_CATEGORY_LABELS)
    )

    generated = frame[frame["split"] == "generated"]
    unknown_labels = sorted(
        generated.loc[generated["simple_category_label"].isna(), "label"]
        .astype(str)
        .unique()
    )
    if unknown_labels:
        raise ValueError(f"Unknown category labels for {model}: {unknown_labels}")
    return frame


def plot_amd_umap_model_scatter_grid() -> None:
    frames = {
        model: train_fit_amd_umap_scatter_frame(model)
        for model in MODEL_COMPARISON_UMAP_MODELS
    }
    x_limits = projection_limits_for_frames(frames, "x")
    y_limits = projection_limits_for_frames(frames, "y")

    density = train_projection_density(
        next(iter(frames.values())),
        x_limits=x_limits,
        y_limits=y_limits,
    )

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(FIG_WIDTH, FIG_WIDTH),
        sharex=True,
        sharey=True,
    )
    fig.subplots_adjust(hspace=0.3)
    flat_axes = np.ravel(axes)
    for ax, model in zip(flat_axes, MODEL_COMPARISON_UMAP_MODELS, strict=True):
        frame = frames[model]
        generated = generated_metastable_smact_valid_projection(frame)
        add_train_projection_density(
            ax,
            frame,
            x_limits=x_limits,
            y_limits=y_limits,
            density=density,
        )
        add_umap_scatter(ax, generated)
        ax.set_title(MODEL_DISPLAY_NAMES.get(model, model), fontsize=FONT_LL)
        ax.set_xlabel("UMAP 1", fontsize=FONT_L, labelpad=PAD_S)
        ax.set_ylabel("UMAP 2", fontsize=FONT_L, labelpad=PAD_S)
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    category_handles = umap_category_handles()
    crystal_handles = crystal_system_handles()
    fig.legend(
        category_handles,
        [handle.get_label() for handle in category_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.05),
        ncol=len(category_handles),
        frameon=False,
        fontsize=FONT_L,
        handletextpad=0,
        columnspacing=1.0,
    )
    fig.legend(
        crystal_handles,
        [handle.get_label() for handle in crystal_handles],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=len(crystal_handles),
        frameon=False,
        fontsize=FONT_L,
        handletextpad=0,
        columnspacing=0.8,
    )

    MODEL_COMPARISON_UMAP_SCATTER_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        MODEL_COMPARISON_UMAP_SCATTER_OUTPUT_PATH,
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()


plot_amd_umap_model_scatter_grid()

## AMD UMAP train-density likelihood by generated model

Train-fit AMD UMAP projected train-density likelihood distributions for metastable SMACT-valid samples from DiffCSP++, WyckoffTransformer, Crystalite, and Chemeleon2.


In [ ]:
AMD_UMAP_MODEL_GRID_LIKELIHOOD_OUTPUT_PATH = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / "amd_umap_projected_train_density_likelihood_models_2x2.pdf"
)


def plot_amd_umap_train_density_likelihood_model_grid() -> None:
    frames = {
        model: train_fit_amd_umap_likelihood_frame(model)
        for model in MODEL_COMPARISON_UMAP_MODELS
    }
    lower, upper = amd_umap_likelihood_bounds(frames)
    x_grid = np.linspace(lower, upper, AMD_UMAP_LIKELIHOOD_GRID_SIZE)

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(FIG_WIDTH, FIG_WIDTH * 0.8),
        sharex=True,
        sharey=True,
    )
    fig.subplots_adjust(hspace=0.3)
    flat_axes = np.ravel(axes)
    for ax, model in zip(flat_axes, MODEL_COMPARISON_UMAP_MODELS, strict=True):
        frame = frames[model]
        add_amd_umap_likelihood_panel(ax, frame, x_grid)

        ax.set_title(MODEL_DISPLAY_NAMES.get(model, model), fontsize=FONT_LL)
        ax.tick_params(axis="both", labelsize=FONT_M, pad=PAD_SS)
        ax.tick_params(axis="y", labelleft=True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for ax in axes[-1, :]:
        ax.set_xlabel("Train KDE log-likelihood", fontsize=FONT_L, labelpad=PAD_S)

    for ax in axes[:, 0]:
        ax.set_ylabel("Density", fontsize=FONT_L, labelpad=PAD_S)

    legend_handles = amd_umap_likelihood_legend_handles()
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.1),
        frameon=False,
        fontsize=FONT_L,
        ncol=len(legend_handles),
    )

    AMD_UMAP_MODEL_GRID_LIKELIHOOD_OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    fig.savefig(
        AMD_UMAP_MODEL_GRID_LIKELIHOOD_OUTPUT_PATH,
        bbox_inches="tight",
    )
    plt.show()


plot_amd_umap_train_density_likelihood_model_grid()

## Top6 volume change and substitution cost by crystal system

For MatterGen and MP20 test, summarize top6 substituted-train candidates among metastable SMACT-valid samples. Top6 means top3 SM-anon plus top3 Wyckoff candidates.


In [ ]:
JOURNAL_TOP6_MODELS = ["mattergen", "test"]
JOURNAL_TOP6_PER_SOURCE = 3
JOURNAL_TOP6_PATH_KEYS = (
    "top3_sm_anon",
    "top3_wyckoff",
    "relaxed_sm_anon_entries",
    "relaxed_wyckoff_entries",
)
JOURNAL_TOP6_SOURCES = (
    ("sm_anon", "top3_sm_anon", "relaxed_sm_anon_entries"),
    ("wyckoff", "top3_wyckoff", "relaxed_wyckoff_entries"),
)
JOURNAL_TOP6_MODEL_COLORS = {
    "mattergen": PALETTE[0],
    "test": PALETTE[1],
}
JOURNAL_TOP6_FIGURE_PATH = (
    ANALYSIS_RESULTS_DIR
    / "journal"
    / (
        "top6_volume_change_and_substitution_cost_by_crystal_system_"
        "mattergen_mp20_test.pdf"
    )
)


def journal_top6_trimmed_mean(values: pd.Series, trim_fraction: float = 0.005) -> float:
    sorted_values = values.dropna().sort_values().to_numpy(dtype=float)
    if len(sorted_values) == 0:
        return float("nan")

    trim_count = int(np.floor(len(sorted_values) * trim_fraction))
    if trim_count == 0 or len(sorted_values) <= 2 * trim_count:
        return float(np.mean(sorted_values))
    return float(np.mean(sorted_values[trim_count:-trim_count]))


def journal_top6_abs_fractional_volume_change(entry: Any, relaxed_entry: Any) -> float:
    volume_per_atom = float(entry.structure.volume) / len(entry.structure)
    relaxed_volume_per_atom = float(relaxed_entry.structure.volume) / len(
        relaxed_entry.structure
    )
    if volume_per_atom == 0.0:
        return float("nan")
    return abs(relaxed_volume_per_atom - volume_per_atom) / volume_per_atom


def require_journal_top6_paths(model: str) -> dict[str, Path]:
    paths = required_paths(model, INPUT_DIR, RAW_RESULTS_DIR, JOURNAL_TOP6_PATH_KEYS)
    missing = missing_required_paths(paths, model=model)
    if missing:
        missing_lines = "\n".join(
            f"- {record['kind']}: {record['path']}" for record in missing
        )
        raise FileNotFoundError(f"Missing {model} top6 input files:\n{missing_lines}")
    return paths


def validate_journal_top6_entry_pair(
    source: str,
    entry: Any,
    relaxed_entry: Any,
) -> None:
    for attr in ("gen_idx", "train_idx", "rank"):
        if int(getattr(entry, attr)) != int(getattr(relaxed_entry, attr)):
            raise ValueError(f"{source}: {attr} mismatch for top6 entry")
    for attr in ("cost_uniform", "cost_mod_petti"):
        if not np.isclose(
            float(getattr(entry, attr)), float(getattr(relaxed_entry, attr))
        ):
            raise ValueError(f"{source}: {attr} mismatch for top6 entry")


def load_journal_model_top6_candidates(
    model: str,
    selected: pd.DataFrame,
) -> pd.DataFrame:
    columns = [
        "model",
        "gen_idx",
        "crystal_system",
        "source",
        "rank",
        "train_idx",
        "cost_mod_petti",
        "train_relaxed_abs_volume_change",
    ]
    if selected.empty:
        return pd.DataFrame(columns=columns)

    paths = require_journal_top6_paths(model)
    crystal_system_by_gen = {
        int(row.gen_idx): str(row.crystal_system)
        for row in selected[["gen_idx", "crystal_system"]].itertuples(index=False)
    }
    rows = []
    for source, entry_key, relaxed_entry_key in JOURNAL_TOP6_SOURCES:
        entries = load_pickle_gz(paths[entry_key])
        relaxed_entries = load_pickle_gz(paths[relaxed_entry_key])
        if len(entries) != len(relaxed_entries):
            raise ValueError(
                f"{model} {source}: top entries length {len(entries)} != "
                f"relaxed entries length {len(relaxed_entries)}"
            )
        for entry, relaxed_entry in zip(entries, relaxed_entries, strict=True):
            validate_journal_top6_entry_pair(source, entry, relaxed_entry)
            gen_idx = int(entry.gen_idx)
            crystal_system = crystal_system_by_gen.get(gen_idx)
            if crystal_system is None:
                continue
            rows.append(
                {
                    "model": model,
                    "gen_idx": gen_idx,
                    "crystal_system": crystal_system,
                    "source": source,
                    "rank": int(entry.rank),
                    "train_idx": int(entry.train_idx),
                    "cost_mod_petti": float(entry.cost_mod_petti),
                    "train_relaxed_abs_volume_change": (
                        journal_top6_abs_fractional_volume_change(entry, relaxed_entry)
                    ),
                }
            )
    return pd.DataFrame(rows, columns=columns)


def build_journal_top6_per_generated(classifications: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "model",
        "gen_idx",
        "crystal_system",
        "top6_match_count",
        "avg_top6_cost_mod_petti",
        "avg_top6_train_relaxed_abs_volume_change",
    ]
    if classifications.empty:
        return pd.DataFrame(columns=columns)

    selected = classifications[
        classifications["model"].isin(JOURNAL_TOP6_MODELS)
        & classifications["is_metastable_smact_valid"]
    ].copy()
    if selected.empty:
        return pd.DataFrame(columns=columns)

    candidates = pd.concat(
        [
            load_journal_model_top6_candidates(
                model,
                selected[selected["model"] == model],
            )
            for model in JOURNAL_TOP6_MODELS
        ],
        ignore_index=True,
    )
    if candidates.empty:
        return pd.DataFrame(columns=columns)

    assert candidates["crystal_system"].isin(CRYSTAL_SYSTEM_ORDER).all()
    per_generated = (
        candidates.groupby(["model", "gen_idx", "crystal_system"], observed=False)
        .agg(
            top6_match_count=("train_idx", "count"),
            avg_top6_cost_mod_petti=("cost_mod_petti", "mean"),
            avg_top6_train_relaxed_abs_volume_change=(
                "train_relaxed_abs_volume_change",
                "mean",
            ),
        )
        .reset_index()
    )
    assert (per_generated["top6_match_count"] <= 2 * JOURNAL_TOP6_PER_SOURCE).all()
    per_generated["crystal_system"] = pd.Categorical(
        per_generated["crystal_system"],
        categories=CRYSTAL_SYSTEM_ORDER,
        ordered=True,
    )
    return per_generated[columns]


def build_journal_top6_metric_summary(per_generated: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "model",
        "crystal_system",
        "generated_with_top6",
        "mean_top6_cost_mod_petti",
        "trimmed_mean_top6_train_relaxed_abs_volume_change",
    ]
    if per_generated.empty:
        return pd.DataFrame(columns=columns)

    matched = per_generated[per_generated["top6_match_count"] > 0].copy()
    if matched.empty:
        return pd.DataFrame(columns=columns)
    return (
        matched.groupby(["model", "crystal_system"], observed=False)
        .agg(
            generated_with_top6=("gen_idx", "count"),
            mean_top6_cost_mod_petti=("avg_top6_cost_mod_petti", "mean"),
            trimmed_mean_top6_train_relaxed_abs_volume_change=(
                "avg_top6_train_relaxed_abs_volume_change",
                journal_top6_trimmed_mean,
            ),
        )
        .reset_index()[columns]
    )


def add_journal_top6_grouped_bars(
    ax: plt.Axes,
    summary: pd.DataFrame,
    *,
    metric_col: str,
    ylabel: str,
) -> None:
    table = summary.pivot(
        index="crystal_system", columns="model", values=metric_col
    ).reindex(index=CRYSTAL_SYSTEM_ORDER, columns=JOURNAL_TOP6_MODELS)
    x = np.arange(len(CRYSTAL_SYSTEM_ORDER))
    width = 0.36
    offsets = np.linspace(
        -width * (len(JOURNAL_TOP6_MODELS) - 1) / 2,
        width * (len(JOURNAL_TOP6_MODELS) - 1) / 2,
        len(JOURNAL_TOP6_MODELS),
    )
    for offset, model in zip(offsets, JOURNAL_TOP6_MODELS, strict=True):
        values = table[model].to_numpy(dtype=float)
        ax.bar(
            x + offset,
            values,
            width=width,
            color=JOURNAL_TOP6_MODEL_COLORS[model],
            edgecolor=WHITE,
            linewidth=0.5,
            label=MODEL_DISPLAY_NAMES.get(model, model),
        )

    ax.set_xticks(x, CRYSTAL_SYSTEM_ORDER, rotation=40, ha="right")
    ax.set_ylabel(ylabel, fontsize=FONT_L, labelpad=PAD_S)
    ax.tick_params(axis="both", labelsize=FONT_M, pad=PAD_SS)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_journal_top6_crystal_system_metrics(summary: pd.DataFrame) -> None:
    if summary.empty:
        display(Markdown("No top6 crystal-system metrics available."))
        return

    plot_data = summary.dropna(
        subset=[
            "mean_top6_cost_mod_petti",
            "trimmed_mean_top6_train_relaxed_abs_volume_change",
        ],
        how="all",
    )
    if plot_data.empty:
        display(Markdown("No non-null top6 crystal-system metrics available."))
        return

    fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_WIDTH * 0.5))
    add_journal_top6_grouped_bars(
        axes[0],
        plot_data,
        metric_col="mean_top6_cost_mod_petti",
        ylabel="Substitution cost",
    )
    add_journal_top6_grouped_bars(
        axes[1],
        plot_data,
        metric_col="trimmed_mean_top6_train_relaxed_abs_volume_change",
        ylabel="Fractional vpa change",
    )
    for tag, ax in zip(("(a)", "(b)"), axes, strict=True):
        ax.text(
            -0.2,
            1.1,
            tag,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=FONT_LL,
            fontweight="bold",
            clip_on=False,
        )

    for ax in axes:
        ax.legend(
            loc="upper right",
            ncol=1,
            frameon=False,
            fontsize=FONT_L,
        )
    fig.tight_layout()

    JOURNAL_TOP6_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(JOURNAL_TOP6_FIGURE_PATH, bbox_inches="tight")
    plt.show()


journal_top6_per_generated = build_journal_top6_per_generated(classifications)
journal_top6_metric_summary = build_journal_top6_metric_summary(
    journal_top6_per_generated
)
display(journal_top6_metric_summary.round(4))
plot_journal_top6_crystal_system_metrics(journal_top6_metric_summary)

## MatterGen top substitutions for Substituted samples

Top element substitutions across all matched train structures for metastable SMACT-valid MatterGen samples classified as Substituted.


In [ ]:
SUBSTITUTION_PATH_KEYS = (
    "generated_structures",
    "training_structures",
    "sm_anon_matches",
    "wyckoff_matches",
    "top3_sm_anon",
    "top3_wyckoff",
    "relaxed_sm_anon_infos",
    "relaxed_wyckoff_infos",
    "relaxed_sm_anon_matches",
    "relaxed_wyckoff_matches",
)
TOP_SUBSTITUTION_COLUMNS = [
    "substitution",
    "train_element",
    "gen_element",
    "cost_mod_petti",
    "count",
    "percent",
]


def candidate_key(row: pd.Series | dict[str, Any]) -> tuple[int, int, float, float]:
    return (
        int(row["gen_idx"]),
        int(row["train_idx"]),
        round(float(row["cost_uniform"]), 12),
        round(float(row["cost_mod_petti"]), 12),
    )


def require_mattergen_substitution_paths() -> dict[str, Path]:
    paths = required_paths(
        "mattergen", INPUT_DIR, RAW_RESULTS_DIR, SUBSTITUTION_PATH_KEYS
    )
    missing = missing_required_paths(paths, model="mattergen")
    if missing:
        missing_lines = "\n".join(
            f"- {record['kind']}: {record['path']}" for record in missing
        )
        raise FileNotFoundError(
            "Missing MatterGen substitution input files:\n" + missing_lines
        )
    return paths


def mattergen_substitution_indices(simple_category: str | None = None) -> set[int]:
    selected = mattergen_classifications
    if simple_category is not None:
        selected = selected[selected["simple_category"].astype(str) == simple_category]
    return set(int(idx) for idx in selected["gen_idx"])


def load_mattergen_substitution_candidates(
    paths: dict[str, Path], selected_indices: set[int]
) -> pd.DataFrame:
    anon = entries_to_frame(
        load_pickle_gz(paths["top3_sm_anon"]),
        load_pickle_gz(paths["relaxed_sm_anon_matches"]),
        "anon",
        infos=load_pickle_gz(paths["relaxed_sm_anon_infos"]),
    )
    wyckoff = entries_to_frame(
        load_pickle_gz(paths["top3_wyckoff"]),
        load_pickle_gz(paths["relaxed_wyckoff_matches"]),
        "wyckoff",
        infos=load_pickle_gz(paths["relaxed_wyckoff_infos"]),
    )
    candidates = pd.concat([anon, wyckoff], ignore_index=True)
    return candidates[candidates["gen_idx"].isin(selected_indices)].reset_index(
        drop=True
    )


def load_match_lookup(
    path: Path,
    selected: pd.DataFrame,
) -> tuple[dict[tuple[int, int, float, float], Any], pd.DataFrame]:
    needed_keys = {candidate_key(row) for _, row in selected.iterrows()}
    lookup: dict[tuple[int, int, float, float], Any] = {}
    duplicate_counts: Counter[tuple[int, int, float, float]] = Counter()

    if not needed_keys:
        return lookup, pd.DataFrame()

    for match in load_pickle_gz(path):
        key = (
            int(match.idx1),
            int(match.idx2),
            round(float(match.cost_uniform), 12),
            round(float(match.cost_mod_petti), 12),
        )
        if key not in needed_keys:
            continue
        if key in lookup:
            duplicate_counts[key] += 1
        else:
            lookup[key] = match

    warning_rows = []
    for key in sorted(needed_keys):
        if key not in lookup:
            warning_rows.append(
                {
                    "gen_idx": key[0],
                    "train_idx": key[1],
                    "cost_uniform": key[2],
                    "cost_mod_petti": key[3],
                    "issue": "missing original match object",
                    "extra_duplicates": 0,
                }
            )
        elif duplicate_counts[key]:
            warning_rows.append(
                {
                    "gen_idx": key[0],
                    "train_idx": key[1],
                    "cost_uniform": key[2],
                    "cost_mod_petti": key[3],
                    "issue": "duplicate original match key; first used",
                    "extra_duplicates": int(duplicate_counts[key]),
                }
            )

    return lookup, pd.DataFrame(warning_rows)


def substitution_pairs_anon(
    match: Any, gen_structures: list[Any], train_structures: list[Any]
) -> list[tuple[str, str]]:
    gen_species = [site.specie.symbol for site in gen_structures[match.idx1]]
    train_species = [site.specie.symbol for site in train_structures[match.idx2]]

    pairs = []
    if match.s1_supercell:
        for train_atom_idx, gen_supercell_atom_idx in enumerate(match.mapping):
            pairs.append(
                (
                    train_species[train_atom_idx],
                    gen_species[int(gen_supercell_atom_idx) // match.fu],
                )
            )
    else:
        for train_supercell_atom_idx, gen_atom_idx in enumerate(match.mapping):
            pairs.append(
                (
                    train_species[train_supercell_atom_idx // match.fu],
                    gen_species[int(gen_atom_idx)],
                )
            )
    return pairs


def substitution_pairs_wyckoff(
    match: Any, gen_structures: list[Any], train_structures: list[Any]
) -> list[tuple[str, str]]:
    gen_species = [site.specie.symbol for site in gen_structures[match.idx1]]
    train_species = [site.specie.symbol for site in train_structures[match.idx2]]
    return [
        (train_species[train_atom_idx], gen_species[int(gen_atom_idx)])
        for train_atom_idx, gen_atom_idx in enumerate(match.atom_map)
    ]


def count_substitution_pairs(
    paths: dict[str, Path], selected: pd.DataFrame
) -> tuple[Counter[tuple[str, str]], pd.DataFrame]:
    if selected.empty:
        return Counter(), pd.DataFrame()

    selected_by_source = {
        source: selected[selected["source"] == source].drop_duplicates(
            ["gen_idx", "train_idx", "cost_uniform", "cost_mod_petti"]
        )
        for source in SOURCE_ORDER
    }
    anon_lookup, anon_warnings = load_match_lookup(
        paths["sm_anon_matches"], selected_by_source["anon"]
    )
    wyckoff_lookup, wyckoff_warnings = load_match_lookup(
        paths["wyckoff_matches"], selected_by_source["wyckoff"]
    )

    gen_structures = load_pickle_gz(paths["generated_structures"])
    train_structures = load_pickle_gz(paths["training_structures"])
    lookups = {"anon": anon_lookup, "wyckoff": wyckoff_lookup}
    pair_builders = {
        "anon": substitution_pairs_anon,
        "wyckoff": substitution_pairs_wyckoff,
    }

    counts: Counter[tuple[str, str]] = Counter()
    for _, row in selected.iterrows():
        source = str(row["source"])
        match = lookups[source].get(candidate_key(row))
        if match is None:
            continue
        for train_element, gen_element in pair_builders[source](
            match, gen_structures, train_structures
        ):
            if train_element != gen_element:
                counts[(train_element, gen_element)] += 1

    warnings = pd.concat(
        [
            anon_warnings.assign(source="anon"),
            wyckoff_warnings.assign(source="wyckoff"),
        ],
        ignore_index=True,
    )
    return counts, warnings


def top_substitution_table(
    counts: Counter[tuple[str, str]], limit: int
) -> pd.DataFrame:
    total = sum(counts.values())
    rows = [
        {
            "substitution": f"{train_element} -> {gen_element}",
            "train_element": train_element,
            "gen_element": gen_element,
            "cost_mod_petti": subst_cost_mod_petti(train_element, gen_element),
            "count": count,
            "percent": 100 * count / total if total else 0.0,
        }
        for (train_element, gen_element), count in counts.most_common(limit)
    ]
    return pd.DataFrame(rows, columns=TOP_SUBSTITUTION_COLUMNS)


def select_all_matched_substitution_pairs(candidates: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty:
        return candidates.copy()

    matched = candidates[candidates["match"]].copy()
    matched = matched.sort_values(
        [
            "gen_idx",
            "train_idx",
            "cost_mod_petti",
            "cost_uniform",
            "source_order",
            "rank",
            "entry_idx",
        ],
        kind="mergesort",
    )
    return matched.drop_duplicates(["gen_idx", "train_idx"], keep="first").reset_index(
        drop=True
    )


def build_mattergen_substituted_top_substitutions(
    limit: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if mattergen_classifications.empty:
        return pd.DataFrame(columns=TOP_SUBSTITUTION_COLUMNS), pd.DataFrame()

    paths = require_mattergen_substitution_paths()
    matched_pairs = select_all_matched_substitution_pairs(
        load_mattergen_substitution_candidates(
            paths, mattergen_substitution_indices(simple_category="2")
        )
    )
    counts, warnings = count_substitution_pairs(paths, matched_pairs)
    return top_substitution_table(counts, limit), warnings


(
    mattergen_substituted_top_substitutions,
    mattergen_substituted_substitution_warnings,
) = build_mattergen_substituted_top_substitutions()
if not mattergen_substituted_substitution_warnings.empty:
    display(Markdown("### Substituted-only match-object lookup warnings"))
    display(mattergen_substituted_substitution_warnings)

mattergen_substituted_top_substitutions

## MatterGen Wyckoff multiset coverage by space group

Coverage of metastable SMACT-valid MatterGen structures relative to theoretical Wyckoff-letter multisets with at most 20 atoms per primitive unit cell. Bars are indexed by space group number; crystal-system labels mark contiguous space-group ranges for readability.

In [ ]:
WYCKOFF_MAX_ATOMS = 20
MATTERGEN_WYCKOFF_COVERAGE_MODEL = "mattergen_80000"
WYCKOFF_COVERAGE_COLUMNS = [
    "both",
    "train_only",
    "model_only",
    "icsd",
    "theoretical_only",
]
WYCKOFF_COVERAGE_LABELS = {
    "both": "MP20 Train & MatterGen",
    "train_only": "MP20 Train only",
    "model_only": "MatterGen only",
    "icsd": "ICSD",
    "theoretical_only": "Theoretical",
}
WYCKOFF_COVERAGE_COLORS = {
    "both": PALETTE[0],
    "train_only": PALETTE[1],
    "model_only": PALETTE[2],
    "icsd": PALETTE[3],
    "theoretical_only": GRAY,
}

WYCKOFF_PARAMS_PATH = resources.files(prototypes).joinpath(
    "wyckoff-position-params.json.gz"
)
with gzip.open(WYCKOFF_PARAMS_PATH, "rt") as file:
    WYCKOFF_POSITION_PARAMS = json.load(file)


def wyckoff_relabelings_for_spg(spg_num: int) -> list[dict[int, str]]:
    return WYCKOFF_POSITION_RELAB_DICT.get(str(spg_num), []) or [
        {ord(letter): letter for letter in WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]}
    ]


def wyckoff_relabeling_cycles(
    spg_num: int, trans: dict[int, str]
) -> list[tuple[str, ...]]:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    seen: set[str] = set()
    cycles: list[tuple[str, ...]] = []
    for letter in sorted(multiplicities):
        if letter in seen:
            continue
        cycle = []
        current = letter
        while current not in seen:
            if current not in multiplicities:
                raise ValueError(
                    f"SG {spg_num} relabeling maps to unknown letter {current!r}"
                )
            seen.add(current)
            cycle.append(current)
            current = current.translate(trans)
        cycles.append(tuple(cycle))
    return cycles


WYCKOFF_CENTERING_FACTORS = {
    "P": 1,
    "A": 2,
    "B": 2,
    "C": 2,
    "I": 2,
    "R": 3,
    "F": 4,
}


def wyckoff_centering_factor_for_spg(spg_num: int) -> int:
    symbol = sg_symbol_from_int_number(spg_num)
    centering = symbol[0]
    if centering not in WYCKOFF_CENTERING_FACTORS:
        raise ValueError(f"SG {spg_num} has unknown centering {centering!r}")
    return WYCKOFF_CENTERING_FACTORS[centering]


def wyckoff_letter_atom_weight(spg_num: int, letter: str) -> int:
    multiplicity = int(WYCKOFF_MULTIPLICITY_DICT[str(spg_num)][letter])
    factor = wyckoff_centering_factor_for_spg(spg_num)
    if multiplicity % factor != 0:
        raise ValueError(
            f"SG {spg_num} Wyckoff {letter!r} multiplicity {multiplicity} "
            f"is not divisible by centering factor {factor}"
        )
    return multiplicity // factor


def wyckoff_letter_max_counts(
    spg_num: int, max_atoms: int = WYCKOFF_MAX_ATOMS
) -> dict[str, int]:
    multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
    position_params = WYCKOFF_POSITION_PARAMS[str(spg_num)]
    max_counts = {}
    for letter in multiplicities:
        n_free = int(position_params.get(letter, 0))
        weight = wyckoff_letter_atom_weight(spg_num, letter)
        max_counts[letter] = max_atoms // weight if n_free > 0 else 1
    return max_counts


def count_bounded_weighted_solutions_leq(
    weights: list[int], bounds: list[int], max_atoms: int
) -> int:
    dp = [0] * (max_atoms + 1)
    dp[0] = 1
    for weight, bound in zip(weights, bounds, strict=True):
        next_dp = [0] * (max_atoms + 1)
        for total, ways in enumerate(dp):
            if ways == 0:
                continue
            for count in range(bound + 1):
                next_total = total + count * weight
                if next_total > max_atoms:
                    break
                next_dp[next_total] += ways
        dp = next_dp
    return sum(dp[1:])


def theoretical_wyckoff_count_for_spg(
    spg_num: int, max_atoms: int = WYCKOFF_MAX_ATOMS
) -> int:
    max_counts = wyckoff_letter_max_counts(spg_num, max_atoms)
    relabelings = wyckoff_relabelings_for_spg(spg_num)
    fixed_total = 0
    for trans in relabelings:
        cycles = wyckoff_relabeling_cycles(spg_num, trans)
        cycle_weights = [
            sum(wyckoff_letter_atom_weight(spg_num, letter) for letter in cycle)
            for cycle in cycles
        ]
        cycle_bounds = [min(max_counts[letter] for letter in cycle) for cycle in cycles]
        fixed_total += count_bounded_weighted_solutions_leq(
            cycle_weights, cycle_bounds, max_atoms
        )
    if fixed_total % len(relabelings) != 0:
        raise ValueError(f"SG {spg_num} Burnside count is not integral")
    return fixed_total // len(relabelings)


def build_theoretical_wyckoff_counts() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "spg_num": spg_num,
            "crystal_system": crystal_system_from_spg_num(spg_num),
            "theoretical": theoretical_wyckoff_count_for_spg(spg_num),
        }
        for spg_num in range(1, 231)
    )


def wyckoff_key_atom_count(spg_num: int, key: tuple[str, ...]) -> int:
    return sum(wyckoff_letter_atom_weight(spg_num, letter) for letter in key)


def wyckoff_key_occupancy_violations(
    spg_num: int, key: tuple[str, ...], max_atoms: int = WYCKOFF_MAX_ATOMS
) -> list[str]:
    counts = Counter(key)
    max_counts = wyckoff_letter_max_counts(spg_num, max_atoms)
    return sorted(
        letter for letter, count in counts.items() if count > max_counts[letter]
    )


def canonical_observed_wyckoff_key(data) -> tuple[int, tuple[str, ...]]:
    if not data.letter_key:
        raise ValueError("WyckoffData has no letter_key entries")
    spg_num = int(data.spg_num)
    key = min(tuple(letter_key) for letter_key in data.letter_key)
    return spg_num, key


def observed_wyckoff_sets_by_spg(
    data: list[Any],
    *,
    label: str,
    max_atoms: int = WYCKOFF_MAX_ATOMS,
    include_indices: set[int] | None = None,
) -> dict[int, set[tuple[str, ...]]]:
    by_spg: dict[int, set[tuple[str, ...]]] = defaultdict(set)
    dropped_atom_count = 0
    dropped_occupancy = 0
    skipped_by_filter = 0
    skipped_none = 0
    for idx, record in enumerate(data):
        if include_indices is not None and idx not in include_indices:
            skipped_by_filter += 1
            continue
        if record is None:
            skipped_none += 1
            continue
        spg_num, key = canonical_observed_wyckoff_key(record)
        multiplicities = WYCKOFF_MULTIPLICITY_DICT[str(spg_num)]
        invalid_letters = sorted(set(key) - set(multiplicities))
        if invalid_letters:
            raise ValueError(
                f"{label} has invalid SG {spg_num} letters {invalid_letters}"
            )
        if wyckoff_key_atom_count(spg_num, key) > max_atoms:
            dropped_atom_count += 1
            continue
        if wyckoff_key_occupancy_violations(spg_num, key, max_atoms):
            dropped_occupancy += 1
            continue
        by_spg[spg_num].add(key)
    print(
        f"{label}: {len(data):,} records, "
        f"{sum(len(v) for v in by_spg.values()):,} "
        f"unique keys <= {max_atoms} primitive atoms, "
        f"{skipped_by_filter:,} skipped by subset filter, "
        f"{skipped_none:,} skipped None, "
        f"{dropped_atom_count:,} dropped by atom count, "
        f"{dropped_occupancy:,} dropped by occupancy limits"
    )
    return dict(by_spg)


def first_metastable_smact_valid_indices_for_wyckoff_coverage(
    model: str,
    target_count: int,
    wyckoff_data: list[Any],
) -> set[int]:
    paths = required_paths(model, INPUT_DIR, RAW_RESULTS_DIR)
    ehull_relaxed = np.asarray(load_pickle_gz(paths["relaxed_ehull"]), dtype=float)
    relax_infos = load_pickle_gz(paths["relax_infos"])
    with np.load(paths["smact_validity"]) as data:
        if "valid" not in data.files:
            raise ValueError(f"{paths['smact_validity']} does not contain 'valid'")
        smact_valid = np.asarray(data["valid"], dtype=bool)

    lengths = {
        "relaxed_ehull": len(ehull_relaxed),
        "relax_infos": len(relax_infos),
        "smact_validity": len(smact_valid),
        "wyckoff_repr": len(wyckoff_data),
    }
    if len(set(lengths.values())) != 1:
        raise ValueError(f"{model} Wyckoff coverage inputs differ in length: {lengths}")

    relax_converged = np.asarray(
        [
            info is not None and bool(info.get("converged", False))
            for info in relax_infos
        ],
        dtype=bool,
    )
    eligible = np.flatnonzero(
        relax_converged
        & np.isfinite(ehull_relaxed)
        & (ehull_relaxed <= METASTABLE_EHULL_MAX)
        & smact_valid
    )
    if len(eligible) < target_count:
        raise ValueError(
            f"{model} has {len(eligible):,} metastable SMACT-valid samples; "
            f"need {target_count:,}."
        )

    selected = eligible[:target_count]
    print(
        f"{model}: selected first {len(selected):,} of {len(eligible):,} "
        "metastable SMACT-valid samples for Wyckoff coverage"
    )
    return set(int(idx) for idx in selected)


def build_wyckoff_coverage_table(
    train_sets: dict[int, set[tuple[str, ...]]],
    model_sets: dict[int, set[tuple[str, ...]]],
    icsd_sets: dict[int, set[tuple[str, ...]]],
    theory: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    theory_by_spg = theory.set_index("spg_num")["theoretical"].to_dict()
    for spg_num in range(1, 231):
        train = train_sets.get(spg_num, set())
        model = model_sets.get(spg_num, set())
        both = train & model
        train_only = train - model
        model_only = model - train
        observed_union = train | model
        icsd_set = icsd_sets.get(spg_num, set())
        icsd = icsd_set - observed_union
        union_with_icsd = observed_union | icsd_set
        theoretical = int(theory_by_spg[spg_num])
        theoretical_only = theoretical - len(union_with_icsd)
        if theoretical_only < 0:
            raise ValueError(
                f"Observed train/generated/ICSD union for SG {spg_num} exceeds "
                f"theoretical count: {len(union_with_icsd)} > {theoretical}"
            )
        rows.append(
            {
                "model": "mattergen",
                "subset": "metastable_smact_valid",
                "spg_num": spg_num,
                "crystal_system": crystal_system_from_spg_num(spg_num),
                "theoretical": theoretical,
                "both": len(both),
                "train_only": len(train_only),
                "model_only": len(model_only),
                "icsd": len(icsd),
                "theoretical_only": theoretical_only,
                "observed_union": len(observed_union),
            }
        )
    frame = pd.DataFrame(rows)
    assert (frame[WYCKOFF_COVERAGE_COLUMNS].sum(axis=1) == frame["theoretical"]).all()
    assert (
        frame[["both", "train_only", "model_only"]].sum(axis=1)
        == frame["observed_union"]
    ).all()
    return frame


def ratio_frame(
    frame: pd.DataFrame, columns: list[str], denominator: str
) -> pd.DataFrame:
    ratios = frame.copy()
    safe_denominator = ratios[denominator].where(ratios[denominator] != 0, 1)
    for column in columns:
        ratios[column] = ratios[column] / safe_denominator
    return ratios


def crystal_system_spans() -> list[tuple[str, int, int]]:
    spans = []
    start = 1
    current = crystal_system_from_spg_num(start)
    for spg_num in range(2, 231):
        system = crystal_system_from_spg_num(spg_num)
        if system != current:
            spans.append((current, start, spg_num - 1))
            start = spg_num
            current = system
    spans.append((current, start, 230))
    return spans


def build_wyckoff_coverage_summary(coverage: pd.DataFrame) -> pd.DataFrame:
    if coverage.empty:
        return pd.DataFrame(
            columns=[
                "crystal_system",
                "theoretical",
                "train and MatterGen",
                "train only",
                "MatterGen only",
                "ICSD",
                "Theoretical",
                "observed union",
                "observed/theoretical",
                "MatterGen-only/theoretical",
            ]
        )

    grouped = (
        coverage.groupby("crystal_system", observed=False)[
            ["theoretical", *WYCKOFF_COVERAGE_COLUMNS, "observed_union"]
        ]
        .sum()
        .reindex(CRYSTAL_SYSTEM_ORDER, fill_value=0)
    )
    grouped = grouped[grouped["theoretical"] > 0]
    summary = grouped.rename(
        columns={
            "both": "train and MatterGen",
            "train_only": "train only",
            "model_only": "MatterGen only",
            "icsd": "ICSD",
            "theoretical_only": "Theoretical",
            "observed_union": "observed union",
        }
    )
    summary["observed/theoretical"] = summary["observed union"] / summary["theoretical"]
    summary["MatterGen-only/theoretical"] = (
        summary["MatterGen only"] / summary["theoretical"]
    )
    summary = summary.reset_index()

    count_columns = [
        "theoretical",
        "train and MatterGen",
        "train only",
        "MatterGen only",
        "ICSD",
        "Theoretical",
        "observed union",
    ]
    summary[count_columns] = summary[count_columns].astype(int)
    return summary


def plot_wyckoff_coverage_by_spg(coverage: pd.DataFrame) -> None:
    if coverage.empty:
        display(Markdown("No MatterGen Wyckoff coverage available."))
        return

    ratios = ratio_frame(coverage, WYCKOFF_COVERAGE_COLUMNS, "theoretical").sort_values(
        "spg_num"
    )
    x = ratios["spg_num"].to_numpy(dtype=int)

    fig, ax = plt.subplots(figsize=(12, 4.5))
    legend_handles, bottom = add_stacked_bars(
        ax,
        x,
        ratios,
        WYCKOFF_COVERAGE_COLUMNS,
        WYCKOFF_COVERAGE_COLORS,
        WYCKOFF_COVERAGE_LABELS,
        width=1.0,
        edgecolor="face",
        linewidth=0.1,
        antialiased=False,
        snap=True,
    )

    assert np.allclose(bottom, 1.0)
    spans = crystal_system_spans()
    centers = [(start + end) / 2 for _, start, end in spans]
    for _, _, end in spans[:-1]:
        ax.axvline(
            end + 0.5,
            color=BLACK,
            alpha=0.75,
            linestyle=(0, (12, 8)),
            linewidth=0.25,
            zorder=3,
        )

    ax.set_xlim(0.5, 230.5)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Ratio", fontsize=20, labelpad=5)
    ax.set_xlabel("Space group", fontsize=20, labelpad=50)
    ax.set_xticks(centers)
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0, pad=4)
    ax.tick_params(axis="y", labelsize=14, pad=4)
    label_offsets = {"triclinic": -0.06, "monoclinic": -0.25}
    for system, start, end in spans:
        ax.text(
            (start + end) / 2,
            label_offsets.get(system, -0.06),
            f"{system}\n{start}-{end}",
            ha="center",
            va="top",
            fontsize=14,
            transform=ax.get_xaxis_transform(),
            clip_on=False,
        )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(
        handles=[legend_handles[column] for column in WYCKOFF_COVERAGE_COLUMNS],
        labels=[WYCKOFF_COVERAGE_LABELS[column] for column in WYCKOFF_COVERAGE_COLUMNS],
        loc="lower center",
        bbox_to_anchor=(0.5, 0.98),
        frameon=False,
        fontsize=14,
        ncol=5,
    )
    fig.tight_layout()

    plot_dir = ANALYSIS_RESULTS_DIR / "journal"
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        plot_dir / "wyckoff_coverage_by_spg.pdf",
        bbox_inches="tight",
    )
    plt.show()


mattergen_wyckoff_theory = build_theoretical_wyckoff_counts()
mattergen_train_wyckoff_path = RAW_RESULTS_DIR / "train" / WYCKOFF_REPR_FILE
mattergen_icsd_wyckoff_path = RAW_RESULTS_DIR / "icsd" / WYCKOFF_REPR_FILE
mattergen_wyckoff_path = (
    RAW_RESULTS_DIR / MATTERGEN_WYCKOFF_COVERAGE_MODEL / WYCKOFF_REPR_FILE
)
mattergen_train_wyckoff_data = load_pickle_gz(mattergen_train_wyckoff_path)
mattergen_icsd_wyckoff_data = load_pickle_gz(mattergen_icsd_wyckoff_path)
mattergen_wyckoff_data = load_pickle_gz(mattergen_wyckoff_path)
mattergen_train_wyckoff_sets = observed_wyckoff_sets_by_spg(
    mattergen_train_wyckoff_data,
    label="train",
)
mattergen_icsd_wyckoff_sets = observed_wyckoff_sets_by_spg(
    mattergen_icsd_wyckoff_data,
    label="icsd",
)
mattergen_metastable_smact_valid_indices = (
    first_metastable_smact_valid_indices_for_wyckoff_coverage(
        MATTERGEN_WYCKOFF_COVERAGE_MODEL,
        len(mattergen_train_wyckoff_data),
        mattergen_wyckoff_data,
    )
)
mattergen_wyckoff_sets = observed_wyckoff_sets_by_spg(
    mattergen_wyckoff_data,
    label=MATTERGEN_WYCKOFF_COVERAGE_MODEL,
    include_indices=mattergen_metastable_smact_valid_indices,
)
mattergen_wyckoff_coverage = build_wyckoff_coverage_table(
    mattergen_train_wyckoff_sets,
    mattergen_wyckoff_sets,
    mattergen_icsd_wyckoff_sets,
    mattergen_wyckoff_theory,
)
mattergen_wyckoff_coverage_summary = build_wyckoff_coverage_summary(
    mattergen_wyckoff_coverage
)
display(mattergen_wyckoff_coverage_summary.round(4))
plot_wyckoff_coverage_by_spg(mattergen_wyckoff_coverage)